# KAPPA: RSNA knee abnormality detection, offline inference

This notebook produces the submission for the RSNA Knee Abnormality Detection competition. It
scores every test study with ten checkpoints, five folds each of two ConvNeXt-Small models, and
writes their per-column rank average to `submission.csv`.

| model | slots | folds | 5-fold OOF macro AUC |
|---|---|---|---|
| `v2_fs_three_planes_convnext` | 3 (fluid-sensitive axial, coronal, sagittal) | 0-4 | 0.8415 |
| `v3_all_slots_convnext` | 6 (every plane x contrast) | 0-4 | 0.8347 |

**Result:** 0.871 on the public leaderboard and 0.850 on the private leaderboard.

## Inputs

* **Competition data:** `test.csv`, `test_series.csv` and the test DICOMs.
* **Model weights:** the `fold*_seed0.pt` checkpoints of both models, attached as a Kaggle Model
  or Dataset that keeps the two model directories. The mount is found automatically under
  `/kaggle/input`; set `WEIGHTS_DIR` in the Config cell if the search picks the wrong one.

## Offline constraint

Kaggle runs this notebook with internet disabled, so it installs nothing and imports no `kappa`
package. It needs only `torch`, `timm`, `pydicom`, `numpy` and `pandas`, which ship in the Kaggle
image. The preprocessing and model code is copied from `src/kappa/`, because the checkpoints must
load with `strict=True` and see the same pixels they were trained on. Each checkpoint stores its
own slots, crop and normalisation, so none of these are hardcoded here.

## Pipeline

1. Build a manifest from `test_series.csv`, reading one DICOM header per series.
2. Decode each chunk of studies into a `.npy` slice cache on `/tmp`.
3. Score every checkpoint on the chunk, averaging three test-time views (identity and ±10°
   rotations).
4. Rank-blend the members, then write and validate `submission.csv`.

A placeholder `submission.csv` is written before scoring and refreshed after every chunk, and
elapsed time is checked against the 9 h (32,400 s) limit as the run goes. All settings are in
the Config cell; `LIMIT` is for smoke tests and must be `None` for a real submission.

## 1. Config

Every tunable is here. Nothing below this cell needs editing for a normal run.

In [ ]:
from pathlib import Path

# --- Where the competition data is mounted -----------------------------------------
# This competition mounts under /kaggle/input/competitions/<slug>/, not the classic
# /kaggle/input/<slug>/. Both are tried.
# None => auto-discover by looking for a directory that holds test_series.csv.
COMP_DIR = None
COMP_DIR_CANDIDATES = (
    "/kaggle/input/competitions/rsna-knee-abnormality-detection",
    "/kaggle/input/rsna-knee-abnormality-detection",
)

# Directory holding <study_uid>/<series_uid>/*.dcm for the test split. None => COMP_DIR
# is searched for a directory named like the series CSV ("test_series.csv" ->
# "test_series/"), which mirrors the layout of the train split.
SERIES_ROOT = None

# --- Where the checkpoints are mounted ----------------------------------------------
# None => auto-discover via WEIGHTS_DIR_CANDIDATES, then a depth-bounded walk.
WEIGHTS_DIR = None
# A Kaggle Model mounts deeper than a dataset:
#   /kaggle/input/models/<owner>/<model>/<framework>/<variation>/<version>/<uploaded dir>
# e.g. /kaggle/input/models/muhammadobaidullaha/kappa-1/pytorch/default/1/models
# These are globs tried in order, so a new version or a renamed variation still resolves.
WEIGHTS_DIR_CANDIDATES = (
    "/kaggle/input/models/*/*/*/*/*/models",   # upload of the models folder
    "/kaggle/input/models/*/*/*/*/*",          # upload of its contents
    "/kaggle/input/*/models",                  # dataset upload of the folder
    "/kaggle/input/*",                         # dataset upload of the contents
)

# Relative to WEIGHTS_DIR. Order does not matter: rank_blend is a mean over members.
# Slots, crop, normalisation and n_slices are read from each checkpoint, so adding a
# member here is the only edit an ensemble change needs.
CHECKPOINTS = tuple(
    f"{cell}/fold{fold}_seed0.pt"
    for cell in ("v2_fs_three_planes_convnext", "v3_all_slots_convnext")
    for fold in range(5)
)
# Refuse to score a partial ensemble by default: a silently dropped member changes the
# blend with no visible sign.
REQUIRE_ALL_CHECKPOINTS = True

# --- Scratch and output --------------------------------------------------------------
# Keep the cache on /tmp: /kaggle/working is the output mount and is capped at ~20 GB.
# The cache takes ~16.8 MB per series (~92 MB per study), so on the hidden test set it
# would fill /kaggle/working at roughly 200 studies, before submission.csv is written.
CACHE_DIR = Path("/tmp/slice_cache")             # <study_uid>/<series_uid>.npy, uint8
OUT_PATH = Path("/kaggle/working/submission.csv")   # must be exactly this name

# Studies per chunk. The cache is built, scored and deleted one chunk at a time, so peak
# disk is CHUNK_STUDIES x ~92 MB rather than the whole test set. 100 => ~9 GB.
# Every study is still decoded exactly once: chunks are disjoint.
CHUNK_STUDIES = 100
# Write a valid submission.csv before scoring anything, and refresh it after every chunk.
# A notebook that dies at chunk 7 of 9 then still leaves a scoreable file instead of none.
WRITE_PARTIAL_SUBMISSION = True

# --- Inference ------------------------------------------------------------------------
BATCH_SIZE = 4        # studies per batch. Each study is (n_slots, n_slices, 3, 384, 384).
NUM_WORKERS = 2       # decode/crop is CPU-bound; Kaggle gives ~4 cores.
# --- Test-time augmentation -----------------------------------------------------------
# Views are averaged in logit space. Legal names: identity, rotate_cw, rotate_ccw,
# zoom_in, zoom_out. Flips are refused by name: a horizontal flip swaps medial and
# lateral, and MCL has no LCL column to swap with, so no relabelling makes it correct.
#
# Cost: TTA multiplies the model forward passes by len(TTA); the DICOM decode is paid
# once per chunk either way. The ten-checkpoint blend runs 5 x 48 + 5 x 96 = 720
# slice-stack passes per study (3 and 6 slots x 16 slices), so three views make 2,160.
#
# The efficiency leaderboard also scores runtime, so run that entry with TTA = ().
# () takes a separate code path that is bit-identical to scoring without TTA.
#
# BUDGET_GUARD accounts for TTA from the second chunk on, because it compares measured
# chunk times against the remaining budget. The first chunk is unguarded, so lower
# CHUNK_STUDIES if the time margin is tight.
TTA = ("identity", "rotate_cw", "rotate_ccw")
AMP = "auto"          # "auto" | "bf16" | "fp16" | "off". auto picks bf16 only on sm_80+.
SCORE_TRANSFORM = "rank"   # what a single-member run writes. Ignored when blending.

# --- Cache build ------------------------------------------------------------------------
# Same values as src/kappa/data/dicom.py, and the copied decode functions default to
# them. Changing them changes the pixels the models see.
#
# 512 is the median native in-plane matrix: the loader can downsample but cannot
# recover detail.
CACHE_SIZE = 512
# Slices per series run 11-320, median 30. The cap bounds the worst case; the batch
# contract only samples n_slices (16) from whatever is cached.
MAX_SLICES = 64
# Per-series percentile window, applied over every pixel of every retained slice at once.
WINDOW_PERCENTILES = (1.0, 99.0)
CACHE_OVERWRITE = False   # the build is resumable; True re-decodes everything.
CACHE_PROGRESS_EVERY = 25

# --- Smoke test -----------------------------------------------------------------------
# Score only the first N studies (sorted by UID). Set this back to None before submitting:
# a submission with missing rows scores as an error, and validate_submission flags it.
LIMIT = None

# --- Budget ------------------------------------------------------------------------------
# The Kaggle notebook limit, in seconds. Mirrors kappa.inference.predict.NOTEBOOK_LIMIT_SECONDS.
NOTEBOOK_LIMIT_SECONDS = 32_400
# Stop scoring once the projection says the next chunk would not fit, and blend what is
# already scored. A partial blend that submits beats a full one killed at 9 h.
BUDGET_GUARD = True
# Time reserved for the blend, the write and the validator.
BUDGET_RESERVE_SECONDS = 600

## 2. Environment

Offline environment variables, an import check that fails **now** rather than four hours in,
and a GPU capability probe.

In [ ]:
import os

# Remove a slice_cache left in /kaggle/working by an earlier version of this notebook.
# /kaggle/working persists across interactive sessions and is capped at ~20 GB, so a
# stale cache there takes the space submission.csv needs. The cache belongs on /tmp.
import shutil as _shutil
_stale = Path("/kaggle/working/slice_cache")
if _stale.is_dir():
    _n = sum(1 for _ in _stale.iterdir())
    _shutil.rmtree(_stale, ignore_errors=True)
    print(f"removed a stale slice_cache from the OUTPUT mount ({_n} study dirs reclaimed)")

# Kaggle has no internet. With these set, a stray hub lookup fails at once instead of
# retrying and hanging. Models are built with pretrained=False, so nothing should be
# fetched anyway.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")

import csv
import io
import sys
import time
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Iterable, Literal, Mapping, Sequence

import numpy as np
import pandas as pd

_missing = []
try:
    import torch
    import torch.nn.functional as F
    from torch import Tensor, nn
    from torch.utils.data import DataLoader, Dataset
except Exception as _exc:                      # noqa: BLE001 - reported, then raised below
    _missing.append(f"torch  ({type(_exc).__name__}: {_exc})")
try:
    import timm
except Exception as _exc:                      # noqa: BLE001
    _missing.append(f"timm   ({type(_exc).__name__}: {_exc})")
try:
    import pydicom
except Exception as _exc:                      # noqa: BLE001
    _missing.append(f"pydicom({type(_exc).__name__}: {_exc})")

if _missing:
    raise ImportError(
        "This notebook needs torch, timm and pydicom, and they must already be in the "
        "image -- there is no internet here, so pip cannot fetch them.\n  missing:\n    "
        + "\n    ".join(_missing)
        + "\n\nFix: attach an offline wheel dataset and `pip install --no-index "
        "--find-links /kaggle/input/<wheels> timm pydicom` in a cell ABOVE this one. "
        "Failing here costs seconds; failing at the first forward pass costs the run."
    )

print(f"python  : {sys.version.split()[0]}")
print(f"torch   : {torch.__version__}")
print(f"timm    : {timm.__version__}")
print(f"pydicom : {pydicom.__version__}")
print(f"numpy   : {np.__version__}   pandas: {pd.__version__}")

In [ ]:
# --- Device and autocast dtype ---------------------------------------------------------
# Kaggle's T4 (sm_75) and P100 (sm_60) have no native bf16, which needs sm_80 or newer.
# Training used bf16, but hardcoding it here would raise or fall back to slow emulation,
# so the dtype is probed.

def resolve_device(device=None):
    """Pick a device, defaulting to CUDA only when it is actually usable.

    Copied from kappa.inference.predict.resolve_device (predict.py:274). It is
    torch.cuda.is_available() rather than a build check on purpose: a CUDA wheel can
    import fine and still report False, and that is exactly the case that must fall back
    to CPU instead of raising halfway through a forward pass.
    """
    if device is not None:
        return torch.device(device)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _bf16_is_real(device):
    """True only when bf16 is native, not emulated.

    torch.cuda.is_bf16_supported() defaults to including_emulation=True on recent torch,
    so it answers True on a T4 and then runs an emulation that is slower than fp16. The
    compute-capability check is the one that answers the question actually being asked.
    """
    if device.type != "cuda":
        return False
    try:
        major, _minor = torch.cuda.get_device_capability(device)
    except Exception:                          # noqa: BLE001 - no device, no bf16
        return False
    return major >= 8


DEVICE = resolve_device(None)
if DEVICE.type == "cuda":
    _name = torch.cuda.get_device_name(DEVICE)
    _cap = torch.cuda.get_device_capability(DEVICE)
    _mem = torch.cuda.get_device_properties(DEVICE).total_memory / 1e9
    print(f"device  : {DEVICE}  {_name}  sm_{_cap[0]}{_cap[1]}  {_mem:.1f} GB")
else:
    print(f"device  : {DEVICE}   (no CUDA -- this will not finish in 9 h on the full test set)")

if AMP == "auto":
    if DEVICE.type != "cuda":
        AMP_RESOLVED = "off"
    elif _bf16_is_real(DEVICE):
        AMP_RESOLVED = "bf16"
    else:
        AMP_RESOLVED = "fp16"
else:
    AMP_RESOLVED = str(AMP)
print(f"amp     : {AMP_RESOLVED}   (requested {AMP!r}; bf16 native = {_bf16_is_real(DEVICE)})")

## 3. Competition constants

Copied from `src/kappa/constants.py`. The header is graded mechanically: a misspelled or
reordered column scores as an error and burns a submission slot.

**`Baker's` carries an apostrophe and five labels contain a space.** Never rebuild this
list by splitting a string on commas. The `csv` module quotes it correctly; a hand-built
header does not.

In [ ]:
from __future__ import annotations   # so `int | None` annotations parse on any Python 3.9+

# --- src/kappa/constants.py, verbatim ---------------------------------------------------
ID_COLUMN = "StudyInstanceUID"

#: The twelve targets, in the exact order the submission header requires.
#: Scored as macro ROC-AUC, so each contributes exactly 1/12 regardless of how
#: rare it is, and each column is scored independently of the others.
LABELS = (
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
)

N_LABELS = len(LABELS)

#: Full submission header: the study identifier followed by the twelve labels.
SUBMISSION_COLUMNS = (ID_COLUMN, *LABELS)

# --- src/kappa/data/manifest.py, verbatim -----------------------------------------------
#: The six plane x contrast slots, in the order that fixes the ``slot`` index.
#: Fixed across the code base: the loader, the manifest and the model all index the
#: slot axis with these positions, so reordering this tuple silently permutes
#: every cached tensor.
SLOT_ORDER: tuple[tuple[str, int], ...] = (
    ("Axial", 0),
    ("Axial", 1),
    ("Coronal", 0),
    ("Coronal", 1),
    ("Sagittal", 0),
    ("Sagittal", 1),
)

N_SLOTS = len(SLOT_ORDER)

PLANES = ("Axial", "Coronal", "Sagittal")

# The manifest keys studies as `study_uid` while the label table and the submission key
# them as `StudyInstanceUID`. Both spellings are named explicitly to avoid mixing them.
ID_COLUMN_MANIFEST = "study_uid"

assert len(LABELS) == 12 and SUBMISSION_COLUMNS[0] == ID_COLUMN
assert "Baker's" in LABELS, "the apostrophe was lost somewhere between here and the header"
print(f"labels  : {N_LABELS}   slots: {N_SLOTS}   header fields: {len(SUBMISSION_COLUMNS)}")

## 4. DICOM decode and slice cache

Copied from `src/kappa/data/dicom.py`. **`StudyDataset` never opens a DICOM** — it reads
`cache_dir/<study_uid>/<series_uid>.npy`, shape `(S, N, N)` uint8. So the cache has to be
built here first.

Pure `pydicom` + NumPy. `cv2`, `PIL` and `scipy` are deliberately **not** used: the resize
is a hand-written anti-aliased separable triangle filter so the cache build has no
dependency beyond NumPy, and — the reason it matters here — so the pixels this notebook
produces are bit-identical to the ones the models were trained on.

The order of operations is load-bearing:

1. read every header, project `ImagePositionPatient` onto the slice normal, sort ascending
   — **never** by `InstanceNumber` (it disagreed with physical order on 40.8% of series);
2. decimate to `MAX_SLICES` across the full extent;
3. decode only the retained files;
4. window over the whole retained stack at once;
5. letterbox each plane to square, **then** resize.

Windowing must precede padding (or the pad value joins the percentile population) and must
precede resizing (the clip is non-linear).

In [ ]:
from __future__ import annotations

# ===========================================================================================
# src/kappa/data/dicom.py -- copied. CACHE_SIZE / MAX_SLICES / WINDOW_PERCENTILES are
# defined in the Config cell above (same values as dicom.py).
# ===========================================================================================

# macOS writes an AppleDouble sidecar named `._<file>` beside every file on an exFAT
# volume. Those sidecars match `*.dcm`, so an unfiltered glob doubles the slice count and
# hands pydicom a 4 KiB resource fork.
_APPLEDOUBLE_PREFIX = "._"

# MONOCHROME1 means low stored value renders *white*. Rare in MR but not impossible, and an
# un-inverted MONOCHROME1 series is a photographic negative: fluid dark, fat bright, every
# learned filter inverted relative to the rest of the corpus.
_INVERTED_PHOTOMETRIC = "MONOCHROME1"


@dataclass(frozen=True)
class SeriesHeader:
    """Geometry and scanner fields for one series, read from a single file.

    Numeric fields are ``None`` rather than ``0.0`` when the tag is absent or empty,
    because a missing spacing must stay visibly missing -- a 0.0 in that column would flow
    into the crop scale and silently resize the anatomy.
    """

    study_uid: str
    series_uid: str
    patient_id: str
    laterality: str
    manufacturer: str
    model: str
    field_strength: float | None
    rows: int
    cols: int
    spacing: float | None
    thickness: float | None
    n_slices: int


@dataclass(frozen=True)
class SeriesVolume:
    """One decoded series: pixels plus everything needed to audit them."""

    pixels: np.ndarray
    slice_positions: np.ndarray
    sop_uids: tuple[str, ...]
    header: SeriesHeader

    @property
    def n_slices(self) -> int:
        """Slices actually decoded, after the MAX_SLICES cap."""
        return int(self.pixels.shape[0])


def series_files(series_dir: Path | str) -> list[Path]:
    """DICOM files in a series directory, AppleDouble sidecars excluded.

    Sorted by filename, which is a stable enumeration order and **not slice order**.
    Falls back to every regular file when nothing matches ``*.dcm``, because a DICOM is
    identified by its preamble and not by its extension.
    """
    directory = Path(series_dir)
    if not directory.is_dir():
        raise NotADirectoryError(f"{directory} is not a series directory")
    files = sorted(
        path
        for path in directory.glob("*.dcm")
        if path.is_file() and not path.name.startswith(_APPLEDOUBLE_PREFIX)
    )
    if files:
        return files
    return sorted(
        path
        for path in directory.iterdir()
        if path.is_file() and not path.name.startswith(_APPLEDOUBLE_PREFIX)
    )


def slice_normal(orientation: Sequence[float]) -> np.ndarray:
    """Unit normal of the image plane from the six ``ImageOrientationPatient`` cosines."""
    cosines = np.asarray(orientation, dtype=np.float64).reshape(-1)
    if cosines.size != 6:
        raise ValueError(
            f"ImageOrientationPatient must hold 6 direction cosines, got {cosines.size}"
        )
    normal = np.cross(cosines[:3], cosines[3:])
    length = float(np.linalg.norm(normal))
    if length == 0.0:
        raise ValueError(
            "ImageOrientationPatient gives a zero-length slice normal; its two "
            "direction cosines are parallel, so the plane is degenerate"
        )
    return normal / length


def slice_projections(
    positions: Sequence[Sequence[float]], orientation: Sequence[float]
) -> np.ndarray:
    """Signed through-plane coordinate of each ``ImagePositionPatient``, in millimetres.

    This is the one true slice order: a dot product with the plane normal, so it is
    invariant to how the scanner numbered the instances.
    """
    normal = slice_normal(orientation)
    origins = np.asarray(positions, dtype=np.float64).reshape(len(positions), 3)
    return origins @ normal


def select_slice_indices(n_available: int, max_slices: int = MAX_SLICES) -> np.ndarray:
    """Indices to keep from an ordered series, capped at ``max_slices``.

    Decimates at equal intervals across the **full** extent rather than taking a centre
    block: a 320-slice series covers the whole joint, and keeping its middle 64 would
    silently crop the anatomy away.
    """
    if n_available < 1:
        raise ValueError(f"n_available must be >= 1, got {n_available}")
    if max_slices < 1:
        raise ValueError(f"max_slices must be >= 1, got {max_slices}")
    if n_available <= max_slices:
        return np.arange(n_available, dtype=np.int64)
    return np.unique(np.rint(np.linspace(0.0, n_available - 1, max_slices)).astype(np.int64))


def window_to_uint8(
    stack: np.ndarray,
    percentiles: tuple[float, float] = WINDOW_PERCENTILES,
    *,
    invert: bool = False,
) -> np.ndarray:
    """Clip a whole series to its own percentile window and scale to 0-255 float.

    Returns float, not uint8: the caller still has to pad and resize, and doing that in
    float and rounding once at the end avoids stacking two quantisation errors. The window
    is computed over **every pixel of every retained slice at once** -- that is what makes
    it per-series.
    """
    low_pct, high_pct = float(percentiles[0]), float(percentiles[1])
    if not 0.0 <= low_pct < high_pct <= 100.0:
        raise ValueError(f"percentiles must satisfy 0 <= low < high <= 100, got {percentiles}")
    values = np.asarray(stack, dtype=np.float32)
    low, high = (float(v) for v in np.percentile(values, (low_pct, high_pct)))
    if high <= low:
        low, high = float(values.min()), float(values.max())
    if high <= low:
        return np.zeros_like(values)
    # Clipped again after scaling, not only before: (high - low) is a float32 reciprocal,
    # so the top of the range lands at 255.00002 and would round to 256, which wraps to 0
    # on the cast to uint8 -- the brightest pixels of the series would come back black.
    scaled = np.clip((np.clip(values, low, high) - low) * (255.0 / (high - low)), 0.0, 255.0)
    return 255.0 - scaled if invert else scaled


def letterbox(plane: np.ndarray, fill: float = 0.0) -> np.ndarray:
    """Pad a 2-D plane to square, centred, so a later resize is isotropic.

    45 distinct in-plane aspect ratios exist in this corpus, so a direct resize to a
    square scales the two axes by different factors and the same anatomy comes out a
    different shape per site.
    """
    if plane.ndim != 2:
        raise ValueError(f"letterbox expects a 2-D plane, got shape {plane.shape}")
    height, width = plane.shape
    side = max(height, width)
    if height == width:
        return plane
    top = (side - height) // 2
    left = (side - width) // 2
    out = np.full((side, side), fill, dtype=plane.dtype)
    out[top : top + height, left : left + width] = plane
    return out


def _resample_weights(n_in: int, n_out: int) -> tuple[np.ndarray, np.ndarray]:
    """Triangle-filter taps for a 1-D resize, widened when downsampling.

    Widening the filter to the downsample factor is what makes this anti-aliased. At
    ``n_in == n_out`` the taps collapse to the identity, so a native-512 series passes
    through untouched.
    """
    if n_in < 1 or n_out < 1:
        raise ValueError(f"resize needs positive sizes, got {n_in} -> {n_out}")
    scale = n_in / n_out
    support = max(1.0, scale)
    centres = (np.arange(n_out, dtype=np.float64) + 0.5) * scale
    n_taps = int(np.ceil(2.0 * support)) + 1
    left = np.floor(centres - support + 0.5).astype(np.int64)
    index = left[:, None] + np.arange(n_taps, dtype=np.int64)[None, :]
    distance = np.abs((index + 0.5) - centres[:, None]) / support
    weight = np.clip(1.0 - distance, 0.0, None)
    weight /= weight.sum(axis=1, keepdims=True)
    return np.clip(index, 0, n_in - 1), weight.astype(np.float32)


def _resample_axis(values: np.ndarray, axis: int, n_out: int) -> np.ndarray:
    """Resize one axis of an array with the anti-aliased triangle filter."""
    n_in = values.shape[axis]
    if n_in == n_out:
        return values
    index, weight = _resample_weights(n_in, n_out)
    # np.take inserts the index array's two dimensions at `axis`; move them to the front
    # so one einsum handles either axis.
    gathered = np.moveaxis(np.take(values, index, axis=axis), (axis, axis + 1), (0, 1))
    reduced = np.einsum("ot,ot...->o...", weight, gathered, optimize=True)
    return np.moveaxis(reduced, 0, axis).astype(np.float32)


def resize_plane(plane: np.ndarray, size: int) -> np.ndarray:
    """Resize a 2-D plane to ``size`` x ``size`` with an anti-aliased separable filter.

    Pure NumPy on purpose: cv2, PIL and scipy are none of them dependencies of the package
    this came from, and substituting one here would change the pixels the model sees.
    """
    if plane.ndim != 2:
        raise ValueError(f"resize_plane expects a 2-D plane, got shape {plane.shape}")
    if size < 1:
        raise ValueError(f"size must be >= 1, got {size}")
    out = np.asarray(plane, dtype=np.float32)
    out = _resample_axis(out, 0, size)
    return _resample_axis(out, 1, size)


def _first(dataset: object, *names: str, default: object = None) -> object:
    """First present, non-empty attribute among ``names``, else ``default``."""
    for name in names:
        value = getattr(dataset, name, None)
        if value is None or (isinstance(value, str) and not value.strip()):
            continue
        return value
    return default


def _as_float(value: object) -> float | None:
    """Coerce a DICOM value to float; absent, empty and unparseable become None."""
    if value is None:
        return None
    if isinstance(value, (bytes, str)):
        text = value.decode("utf-8", "replace") if isinstance(value, bytes) else value
        text = text.strip()
        if not text:
            return None
        try:
            return float(text)
        except ValueError:
            return None
    if isinstance(value, Sequence):
        return _as_float(value[0]) if len(value) else None
    try:
        return float(value)  # type: ignore[arg-type]
    except (TypeError, ValueError):
        return None


def _as_str(value: object) -> str:
    """Coerce a DICOM value to a stripped string; absent becomes empty."""
    if value is None:
        return ""
    if isinstance(value, bytes):
        return value.decode("utf-8", "replace").strip()
    if not isinstance(value, str) and isinstance(value, Sequence):
        return _as_str(value[0]) if len(value) else ""
    return str(value).strip()


def _header_from_dataset(dataset: object, n_slices: int) -> SeriesHeader:
    """Build a SeriesHeader from one already-read pydicom dataset.

    ``spacing`` is ``PixelSpacing[0]``, the row spacing; in-plane spacing is isotropic in
    this corpus. This value scales the physical crop, so getting it wrong resizes the
    anatomy invisibly.
    """
    return SeriesHeader(
        study_uid=_as_str(getattr(dataset, "StudyInstanceUID", "")),
        series_uid=_as_str(getattr(dataset, "SeriesInstanceUID", "")),
        patient_id=_as_str(getattr(dataset, "PatientID", "")),
        laterality=_as_str(_first(dataset, "Laterality", "ImageLaterality", default="")),
        manufacturer=_as_str(getattr(dataset, "Manufacturer", "")),
        model=_as_str(getattr(dataset, "ManufacturerModelName", "")),
        field_strength=_as_float(getattr(dataset, "MagneticFieldStrength", None)),
        rows=int(_as_float(getattr(dataset, "Rows", 0)) or 0),
        cols=int(_as_float(getattr(dataset, "Columns", 0)) or 0),
        spacing=_as_float(getattr(dataset, "PixelSpacing", None)),
        thickness=_as_float(
            _first(dataset, "SliceThickness", "SpacingBetweenSlices", default=None)
        ),
        n_slices=int(n_slices),
    )


def read_series_header(series_dir: Path | str) -> SeriesHeader:
    """Geometry and scanner fields for a series, reading **one** file and no pixels.

    ``n_slices`` is the file count, not a tag: no DICOM attribute records how many
    instances a series has. Header-only because the manifest build touches every series
    and decoding their pixels would be the entire cost of that pass for none of its value.
    """
    import pydicom

    files = series_files(series_dir)
    if not files:
        raise FileNotFoundError(f"no DICOM files under {Path(series_dir)}")
    dataset = pydicom.dcmread(str(files[0]), stop_before_pixels=True)
    return _header_from_dataset(dataset, n_slices=len(files))


def _slice_pixels(dataset: object) -> np.ndarray:
    """Rescaled float32 pixels of one slice.

    ``pixel_array`` already honours PixelRepresentation and BitsStored; the job here is
    the affine RescaleSlope/RescaleIntercept map, which is absent on much of this corpus
    and must not be assumed to be the identity when it is present.
    """
    pixels = np.asarray(dataset.pixel_array)  # type: ignore[attr-defined]
    if pixels.ndim != 2:
        raise ValueError(
            f"expected a single-frame 2-D image, got shape {pixels.shape}. A "
            "multi-frame instance needs its frames split into slices first, and "
            "silently taking frame 0 would drop the rest of the series."
        )
    slope = _as_float(getattr(dataset, "RescaleSlope", None))
    intercept = _as_float(getattr(dataset, "RescaleIntercept", None))
    out = pixels.astype(np.float32)
    if slope is not None and slope != 1.0:
        out = out * np.float32(slope)
    if intercept is not None and intercept != 0.0:
        out = out + np.float32(intercept)
    return out


def read_series(
    series_dir: Path | str,
    size: int = CACHE_SIZE,
    max_slices: int = MAX_SLICES,
    window: tuple[float, float] = WINDOW_PERCENTILES,
) -> SeriesVolume:
    """Decode a series to ``(S, size, size)`` uint8, ordered by geometry.

    See the markdown above for why the five steps are in this order and cannot be
    reordered.
    """
    import pydicom

    files = series_files(series_dir)
    if not files:
        raise FileNotFoundError(f"no DICOM files under {Path(series_dir)}")

    headers = [pydicom.dcmread(str(path), stop_before_pixels=True) for path in files]
    orientation = _first(headers[0], "ImageOrientationPatient")
    if orientation is None:
        raise ValueError(
            f"{files[0]} has no ImageOrientationPatient, so slice order cannot be "
            "derived from geometry. Falling back to InstanceNumber is not "
            "acceptable: it disagreed with physical order on 49 of 120 series probed "
            "(40.8%)."
        )
    positions = []
    reference = np.asarray([float(value) for value in orientation], dtype=np.float64)
    for path, dataset in zip(files, headers):
        origin = _first(dataset, "ImagePositionPatient")
        if origin is None:
            raise ValueError(f"{path} has no ImagePositionPatient, so it cannot be ordered")
        positions.append([float(value) for value in origin])

        # Every slice is projected onto the first slice's normal, so a slice acquired in a
        # different plane lands at a meaningless position. That reorders the volume
        # silently -- no exception, no shape mismatch, just a stack in the wrong order.
        own = _first(dataset, "ImageOrientationPatient")
        if own is None:
            raise ValueError(f"{path} has no ImageOrientationPatient")
        if not np.allclose(np.asarray([float(v) for v in own], dtype=np.float64),
                           reference, atol=1e-4):
            raise ValueError(
                f"{path} has ImageOrientationPatient {[float(v) for v in own]}, which "
                f"differs from the series' first slice {reference.tolist()}. Slices in "
                "two planes are not one volume, and projecting them onto a single "
                "normal orders them wrongly without failing."
            )
    projections = slice_projections(positions, orientation)

    # All-equal projections mean the geometry says nothing about order, and the stable
    # argsort below would quietly fall back to filename order -- which is the very thing
    # this function exists to avoid.
    if len(projections) > 1 and float(np.ptp(projections)) == 0.0:
        raise ValueError(
            f"{Path(series_dir)}: all {len(projections)} slices project to the same "
            "point on the slice normal, so geometry cannot order them. Sorting by "
            "filename here would be an arbitrary order presented as an anatomical one."
        )

    # Stable sort so that slices sharing a z -- interleaved echoes, a repeated acquisition
    # -- keep filename order instead of shuffling between runs.
    order = np.argsort(projections, kind="stable")
    keep = order[select_slice_indices(len(order), max_slices)]

    invert = _as_str(getattr(headers[0], "PhotometricInterpretation", "")).upper() == (
        _INVERTED_PHOTOMETRIC
    )
    planes = [_slice_pixels(pydicom.dcmread(str(files[index]))) for index in keep]
    shapes = {plane.shape for plane in planes}
    if len(shapes) > 1:
        raise ValueError(
            f"{Path(series_dir)} mixes in-plane matrices {sorted(shapes)}; a series "
            "whose slices are different sizes is not one volume"
        )

    windowed = window_to_uint8(np.stack(planes), window, invert=invert)
    pixels = np.empty((len(planes), size, size), dtype=np.uint8)
    for position, plane in enumerate(windowed):
        resized = resize_plane(letterbox(plane), size)
        pixels[position] = np.clip(np.rint(resized), 0, 255).astype(np.uint8)

    header = _header_from_dataset(headers[0], n_slices=len(files))
    return SeriesVolume(
        pixels=pixels,
        slice_positions=projections[keep].astype(np.float64),
        sop_uids=tuple(_as_str(getattr(headers[index], "SOPInstanceUID", "")) for index in keep),
        header=header,
    )


def study_dirs(series_root: Path | str) -> list[Path]:
    """Study directories under ``<series_root>``, sorted, AppleDouble excluded."""
    root = Path(series_root)
    if not root.is_dir():
        raise NotADirectoryError(f"{root} is not a series root")
    return sorted(
        path
        for path in root.iterdir()
        if path.is_dir() and not path.name.startswith(_APPLEDOUBLE_PREFIX)
    )


@dataclass(frozen=True)
class CacheReport:
    """What one cache-build pass actually did, for a log line and an exit code."""

    n_studies: int
    n_written: int
    n_skipped: int
    n_failed: int
    n_slices: int
    bytes_written: int
    seconds: float
    failures: tuple[tuple[str, str], ...] = ()

    @property
    def n_attempted(self) -> int:
        """Series this pass tried to decode, i.e. excluding the resumed ones."""
        return self.n_written + self.n_failed

    def summary(self) -> str:
        """One-line summary, including the per-series rate the next job is sized from."""
        rate = self.seconds / self.n_written if self.n_written else float("nan")
        return (
            f"{self.n_studies} studies: {self.n_written} written, "
            f"{self.n_skipped} already cached, {self.n_failed} failed; "
            f"{self.n_slices} slices, {self.bytes_written / 1e9:.2f} GB, "
            f"{self.seconds:.1f}s ({rate:.3f} s/series written)"
        )


def cache_series(
    series_dir: Path | str,
    out_path: Path | str,
    *,
    size: int = CACHE_SIZE,
    max_slices: int = MAX_SLICES,
    window: tuple[float, float] = WINDOW_PERCENTILES,
    overwrite: bool = False,
) -> int:
    """Decode one series to ``out_path``. Returns bytes written, or 0 if skipped.

    Written to a temporary name and renamed into place. np.save on a killed process leaves
    a half-written .npy whose header still parses, and the resume check ("does the file
    exist?") would then treat a truncated volume as done. os.replace is atomic, so the
    file either is not there or is complete -- which is what makes a notebook that ran out
    of time restartable.
    """
    target = Path(out_path)
    if target.is_file() and not overwrite:
        return 0
    volume = read_series(series_dir, size=size, max_slices=max_slices, window=window)
    target.parent.mkdir(parents=True, exist_ok=True)
    staging = target.with_name(f".{target.name}.partial")
    try:
        with open(staging, "wb") as handle:
            np.save(handle, volume.pixels, allow_pickle=False)
        os.replace(staging, target)
    finally:
        staging.unlink(missing_ok=True)
    return target.stat().st_size


def build_cache(
    series_root: Path | str,
    cache_root: Path | str,
    *,
    studies: Sequence[str] | None = None,
    shard_index: int = 0,
    shard_count: int = 1,
    size: int = CACHE_SIZE,
    max_slices: int = MAX_SLICES,
    window: tuple[float, float] = WINDOW_PERCENTILES,
    overwrite: bool = False,
    limit: int | None = None,
    progress_every: int = 25,
) -> CacheReport:
    """Cache every series of a shard of studies, skipping what is already cached.

    A series that fails to decode is recorded and the pass continues: one unreadable
    series must not cost the others. The caller decides whether the failure count is
    acceptable, from CacheReport.failures.
    """
    root = Path(series_root)
    cache = Path(cache_root)
    if shard_count < 1:
        raise ValueError(f"shard_count must be >= 1, got {shard_count}")
    if not 0 <= shard_index < shard_count:
        raise ValueError(f"shard_index must be in [0, {shard_count}), got {shard_index}")

    if studies is None:
        selected = study_dirs(root)
    else:
        selected = [root / uid for uid in sorted(set(studies))]
    selected = selected[shard_index::shard_count]
    if limit is not None:
        selected = selected[:limit]

    written = skipped = failed = n_slices = 0
    total_bytes = 0
    failures: list[tuple[str, str]] = []
    started = time.perf_counter()
    for position, study in enumerate(selected):
        if not study.is_dir():
            failures.append((study.name, "no such study directory"))
            failed += 1
            continue
        for series in sorted(
            path
            for path in study.iterdir()
            if path.is_dir() and not path.name.startswith(_APPLEDOUBLE_PREFIX)
        ):
            target = cache / study.name / f"{series.name}.npy"
            if target.is_file() and not overwrite:
                skipped += 1
                continue
            try:
                size_bytes = cache_series(
                    series,
                    target,
                    size=size,
                    max_slices=max_slices,
                    window=window,
                    overwrite=overwrite,
                )
            except Exception as exc:  # noqa: BLE001 - one bad series must not stop the pass
                failures.append((f"{study.name}/{series.name}", f"{type(exc).__name__}: {exc}"))
                failed += 1
                continue
            written += 1
            total_bytes += size_bytes
            # (S, size, size) uint8, so the slice count is recoverable from the file size
            # without re-reading the array.
            n_slices += size_bytes // (size * size)
        if progress_every and position and position % progress_every == 0:
            elapsed = time.perf_counter() - started
            print(
                f"  {position}/{len(selected)} studies, {written} series written, "
                f"{elapsed:.0f}s",
                file=sys.stderr,
                flush=True,
            )

    return CacheReport(
        n_studies=len(selected),
        n_written=written,
        n_skipped=skipped,
        n_failed=failed,
        n_slices=int(n_slices),
        bytes_written=int(total_bytes),
        seconds=time.perf_counter() - started,
        failures=tuple(failures),
    )

## 5. Manifest

`read_series_index` reads `test_series.csv` directly and derives plane / contrast / slot
with **no DICOM access at all**. `build_manifest` then adds one *header-only* read per
series (`stop_before_pixels=True`) to fill `spacing`, `rows` and `cols`.

Those three are not bookkeeping: `cache_mm_per_pixel(spacing, rows, cols, cache_size)` is
what converts the 150 mm physical field of view into a pixel crop. Get them wrong and
every crop is scaled by the aspect ratio, which is invisible in the images.

`StudyDataset` requires exactly `study_uid, series_uid, slot, spacing, rows, cols`. Note
the manifest key is **`study_uid`**, not `StudyInstanceUID`.

**Deviation from source:** `kappa.data.manifest.build_manifest` also emits `patient_id`,
`laterality`, `manufacturer`, `model`, `field_strength` and `scanner_fingerprint`. Those
exist to build scanner-grouped folds; nothing on the inference path reads them, so they
are dropped. Every column that *is* emitted is computed by the identical expression.

In [ ]:
from __future__ import annotations

# ===========================================================================================
# src/kappa/data/manifest.py -- read_series_index and build_manifest, adapted for
# inference (see the note above and the docstrings).
# ===========================================================================================

_SLOT_LOOKUP = {pair: index for index, pair in enumerate(SLOT_ORDER)}

# Column names as the competition writes them. `Fat_Suppression` is read only to be
# checked against `Fluid_Sensitive`.
_PLANE_COLUMN = "Anatomical_Plane"
_FLUID_COLUMN = "Fluid_Sensitive"
_FAT_COLUMN = "Fat_Suppression"


def slot_index(plane: str, fs: int) -> int:
    """Slot id for a (plane, contrast) pair.

    Raises rather than returning a sentinel: a -1 slot would quietly index the last
    element of the slot axis instead.
    """
    try:
        return _SLOT_LOOKUP[(plane, int(fs))]
    except (KeyError, TypeError, ValueError):
        raise ValueError(
            f"no slot for plane={plane!r} fs={fs!r}; expected one of {SLOT_ORDER}"
        ) from None


def read_series_index(series_csv: Path | str) -> pd.DataFrame:
    """Load test_series.csv and derive plane, fs and slot.

    INTENTIONAL DEVIATION FROM src/kappa/data/manifest.py:240. The source raises on an
    unknown plane and on any Fluid_Sensitive/Fat_Suppression disagreement, which is right
    for a training pipeline: silently changing the meaning of the slot axis is worse than
    stopping. It is wrong HERE. This runs in a Kaggle rerun against a hidden test set, and
    a raise scores zero while a dropped series only costs one slot of one study.

    The source's "byte-identical across all 24,371 rows" was measured on train_series.csv.
    The hidden test set is different data, and Kaggle says so explicitly. So every strict
    check below degrades to a counted, printed warning:

      * plane is stripped and matched case-insensitively, so " axial" still resolves;
      * an unresolvable plane drops the series (it becomes an absent slot, never a
        guessed one -- putting a sagittal series in the axial slot is worse than omitting
        it, because the model was never trained to see it there);
      * fs prefers Fluid_Sensitive, the column the slot axis was designed on, and falls
        back to Fat_Suppression only where Fluid_Sensitive is null;
      * a row with no usable contrast bit at all is dropped rather than guessed.

    A study whose every series is dropped still gets a submission row: it is scored
    all-slots-absent, which is verified to yield finite logits.
    """
    frame = pd.read_csv(series_csv)
    missing = {"StudyInstanceUID", "SeriesInstanceUID", _PLANE_COLUMN, _FLUID_COLUMN} - set(
        frame.columns
    )
    if missing:
        # Not recoverable: without these there is no slot axis at all.
        raise ValueError(f"{series_csv} is missing required columns: {sorted(missing)}")

    n_in = len(frame)
    notes: list[str] = []

    # --- contrast bit ------------------------------------------------------------------
    fluid = pd.to_numeric(frame[_FLUID_COLUMN], errors="coerce")
    if _FAT_COLUMN in frame.columns:
        fat = pd.to_numeric(frame[_FAT_COLUMN], errors="coerce")
        both = fluid.notna() & fat.notna()
        disagree = int((both & (fluid != fat)).sum())
        if disagree:
            notes.append(
                f"{disagree} row(s) disagree between {_FLUID_COLUMN} and {_FAT_COLUMN}; "
                f"using {_FLUID_COLUMN}, which is the column the slot axis was built on"
            )
        fluid = fluid.where(fluid.notna(), fat)      # fill only where Fluid is null

    # --- plane -------------------------------------------------------------------------
    canon = {p.strip().casefold(): p for p in PLANES}
    raw_plane = frame[_PLANE_COLUMN].astype(str).str.strip()
    plane = raw_plane.map(lambda v: canon.get(v.casefold()))
    recovered = int(((plane.notna()) & (raw_plane != plane)).sum())
    if recovered:
        notes.append(f"{recovered} plane value(s) recovered by strip/case-fold")

    keep = plane.notna() & fluid.notna()
    dropped = int((~keep).sum())
    if dropped:
        bad = sorted(set(raw_plane[plane.isna()]))[:5]
        notes.append(
            f"DROPPED {dropped} of {n_in} series with an unusable plane or contrast bit "
            f"(plane values seen: {bad}); each becomes an absent slot, never a guessed one"
        )

    out = pd.DataFrame(
        {
            "study_uid": frame["StudyInstanceUID"].astype(str),
            "series_uid": frame["SeriesInstanceUID"].astype(str),
            "plane": plane,
            "fs": fluid,
        }
    )[keep].reset_index(drop=True)
    out["fs"] = out["fs"].astype(int).clip(0, 1)
    out["plane"] = out["plane"].astype(str)
    out["slot"] = [slot_index(p, f) for p, f in zip(out["plane"], out["fs"])]

    for note in notes:
        print(f"  !! series index: {note}")
    print(f"  series index: kept {len(out)} of {n_in} rows")
    return out


# The 10 columns this build_manifest emits, in order. The source tuple has 16; the six
# extra columns (patient_id, laterality, manufacturer, model, field_strength,
# scanner_fingerprint) are only used to build folds and are not declared here, since
# a declared column that is never filled would be all-null.
# StudyDataset requires exactly: study_uid, series_uid, slot, spacing, rows, cols.
MANIFEST_COLUMNS: tuple[str, ...] = (
    "study_uid",
    "series_uid",
    "plane",
    "fs",
    "slot",
    "n_slices",
    "rows",
    "cols",
    "spacing",
    "thickness",
)


def build_manifest(
    series_csv: Path | str,
    series_root: Path | str,
    *,
    studies: Iterable[str] | None = None,
    require_all: bool = False,
    progress_every: int = 200,
) -> pd.DataFrame:
    """One manifest row per series, joining test_series.csv to one DICOM header each.

    ``series_root`` holds ``<study_uid>/<series_uid>/<sop_uid>.dcm``. Series with no
    directory on disk are skipped unless ``require_all``; the returned frame is the honest
    record of what was actually found.
    """
    index = read_series_index(series_csv)
    if studies is not None:
        wanted = set(studies)
        index = index[index["study_uid"].isin(wanted)].reset_index(drop=True)

    root = Path(series_root)

    rows: list[dict[str, Any]] = []
    missing: list[str] = []
    unreadable: list[tuple[str, str]] = []
    started = time.perf_counter()
    for position, record in enumerate(index.itertuples(index=False)):
        series_dir = root / record.study_uid / record.series_uid
        if not series_dir.is_dir():
            missing.append(record.series_uid)
            continue

        # Deviation from src/kappa/data/manifest.py: the header read is guarded. Unguarded,
        # one unreadable series in the hidden test set would abort the manifest build and
        # the whole submission. Guarded, it becomes an absent slot.
        try:
            header = read_series_header(series_dir)
        except Exception as exc:                      # noqa: BLE001 - any decode failure
            unreadable.append((record.series_uid, f"{type(exc).__name__}: {exc}"))
            continue
        rows.append(
            {
                "study_uid": record.study_uid,
                "series_uid": record.series_uid,
                "plane": record.plane,
                "fs": int(record.fs),
                "slot": int(record.slot),
                "n_slices": int(header.n_slices),
                "rows": int(_as_float(header.rows) or 0),
                "cols": int(_as_float(header.cols) or 0),
                "spacing": _as_float(header.spacing),
                "thickness": _as_float(header.thickness),
            }
        )
        if progress_every and position and position % progress_every == 0:
            print(
                f"  {position}/{len(index)} series  {time.perf_counter() - started:.0f}s",
                file=sys.stderr,
                flush=True,
            )

    if missing and require_all:
        raise FileNotFoundError(
            f"{len(missing)} of {len(index)} series have no directory under {root}; "
            f"first missing: {missing[0]}"
        )
    if missing:
        print(
            f"note: skipped {len(missing)} of {len(index)} series with no directory "
            f"under {root}",
            file=sys.stderr,
        )

    if unreadable:
        print(
            f"  !! manifest: {len(unreadable)} of {len(index)} series have a directory but "
            f"no readable DICOM header; each becomes an absent slot. First three:",
            file=sys.stderr,
        )
        for name, why in unreadable[:3]:
            print(f"       {name}: {why}", file=sys.stderr)

    manifest = pd.DataFrame(rows, columns=list(MANIFEST_COLUMNS))
    return manifest.sort_values(["study_uid", "slot", "series_uid"]).reset_index(drop=True)

## 6. Dataset

Copied from `src/kappa/data/dataset.py`, verbatim. This is the half that must be
bit-faithful: a different crop, a different normalisation or a different slice sample and
the backbone is shown a distribution it was never trained on — which still produces a
number.

Three things worth naming:

* **`CropSpec` is physical.** 150 mm at 384 px = 0.390625 mm/px, *identical for every
  series regardless of its native spacing*. `cache_mm_per_pixel` corrects for the fact
  that the cache letterboxed to `max(rows, cols)` before resizing to 512, so the cached
  pixel is **not** `spacing`.
* **`Normalisation` comes out of the checkpoint**, never re-resolved from timm. A model
  whose weights want `mean=0.5` fed ImageNet constants still runs.
* **Absent slots are exact zeros with `present=False`.** A study with no usable series in
  the selected slots is emitted honestly rather than dropped — which is also what
  guarantees every test UID gets a submission row.

In [ ]:
from __future__ import annotations

# ===========================================================================================
# src/kappa/data/dataset.py -- copied verbatim.
# ===========================================================================================


@dataclass(frozen=True)
class Normalisation:
    """Per-channel mean and standard deviation, taken from the backbone.

    A single-element mean/std is broadcast across the three 2.5D channels, which is what
    greyscale-pretrained medical backbones report.
    """

    mean: tuple[float, ...]
    std: tuple[float, ...]

    def __post_init__(self) -> None:
        if not self.mean or not self.std:
            raise ValueError("normalisation mean and std must be non-empty")
        if any(s == 0.0 for s in self.std):
            raise ValueError(f"normalisation std contains zero: {self.std}")

    @classmethod
    def identity(cls) -> "Normalisation":
        """Leave pixels in [0, 1]. For wiring tests, never for a reported score."""
        return cls((0.0,), (1.0,))

    @classmethod
    def from_data_config(cls, config: Mapping[str, Any]) -> "Normalisation":
        """Build from a timm resolved data config, i.e. its ``mean``/``std`` keys."""
        try:
            mean, std = config["mean"], config["std"]
        except KeyError as exc:  # a config without these keys is not a data config
            raise KeyError(f"data config has no {exc} key; got keys {sorted(config)}") from exc
        return cls(tuple(float(v) for v in mean), tuple(float(v) for v in std))

    def tensors(self, channels: int = 3) -> tuple[Tensor, Tensor]:
        """Return ``(mean, std)`` shaped ``(channels, 1, 1)`` for broadcasting."""
        mean = _broadcast(self.mean, channels, "mean")
        std = _broadcast(self.std, channels, "std")
        return mean.view(channels, 1, 1), std.view(channels, 1, 1)


def _broadcast(values: Sequence[float], channels: int, name: str) -> Tensor:
    if len(values) == channels:
        return torch.tensor(values, dtype=torch.float32)
    if len(values) == 1:
        return torch.full((channels,), float(values[0]), dtype=torch.float32)
    raise ValueError(f"normalisation {name} has {len(values)} entries, need 1 or {channels}")


@dataclass(frozen=True)
class CropSpec:
    """A centre crop of ``fov_mm`` millimetres resampled to ``out_px`` pixels.

    The defaults give 150 mm at 384 px, i.e. 0.390625 mm/px, chosen so a knee fills the
    frame at every site's spacing rather than at the median one.
    """

    fov_mm: float = 150.0
    out_px: int = 384

    def __post_init__(self) -> None:
        if self.fov_mm <= 0.0 or self.out_px < 1:
            raise ValueError(f"CropSpec needs fov_mm > 0 and out_px >= 1, got {self}")

    @property
    def mm_per_pixel(self) -> float:
        """Physical size of one output pixel. Identical for every series, by design."""
        return self.fov_mm / self.out_px


def cache_mm_per_pixel(spacing: float, rows: int, cols: int, cache_size: int) -> float:
    """Millimetres per pixel of a cached slice.

    The cache letterboxes each slice to a square of ``max(rows, cols)`` before resizing to
    ``cache_size``, so the cached pixel is *not* ``spacing`` unless the source happened to
    be square at exactly ``cache_size``. Getting this wrong scales every crop by the
    aspect ratio and is invisible in the images.
    """
    if spacing <= 0.0 or rows < 1 or cols < 1 or cache_size < 1:
        raise ValueError(f"bad geometry: spacing={spacing} rows={rows} cols={cols}")
    return float(spacing) * float(max(int(rows), int(cols))) / float(cache_size)


def sample_slice_indices(n_available: int, n_slices: int) -> np.ndarray:
    """``n_slices`` indices at equal intervals across the full geometric extent.

    The cache is already ordered by the projection of ImagePositionPatient onto the slice
    normal, so position in the array *is* geometric order and no re-sorting happens here.
    A series shorter than ``n_slices`` repeats indices rather than padding: a duplicated
    real slice is closer to the truth than a black one.
    """
    if n_available < 1:
        raise ValueError(f"n_available must be >= 1, got {n_available}")
    if n_slices < 1:
        raise ValueError(f"n_slices must be >= 1, got {n_slices}")
    if n_available == 1:
        return np.zeros(n_slices, dtype=np.int64)
    return np.rint(np.linspace(0.0, n_available - 1, n_slices)).astype(np.int64)


def contiguous_slice_indices(n_available: int, n_slices: int) -> np.ndarray:
    """``n_slices`` **adjacent** indices, centred on the middle of the series.

    The volumetric arm's sampler. Kept so that ``slice_mode`` read out of a checkpoint has
    somewhere to resolve to; both checkpoints this notebook ships are "spanning".
    """
    if n_available < 1:
        raise ValueError(f"n_available must be >= 1, got {n_available}")
    if n_slices < 1:
        raise ValueError(f"n_slices must be >= 1, got {n_slices}")
    if n_available <= n_slices:
        # The whole series is the block: same extent an arange-and-clip would give,
        # without the duplicated end slabs.
        return sample_slice_indices(n_available, n_slices)
    # Floor division, so an odd leftover slice goes to the high-index end rather than
    # being split. Deterministic and stated.
    start = (n_available - n_slices) // 2
    return np.arange(start, start + n_slices, dtype=np.int64)


#: The slice samplers, keyed the way the checkpoint's meta spells them.
SLICE_SAMPLERS = {
    "spanning": sample_slice_indices,
    "contiguous": contiguous_slice_indices,
}


def neighbour_indices(centres: Iterable[int], n_available: int) -> np.ndarray:
    """``(len(centres), 3)`` indices of ``(previous, centre, next)`` per centre.

    Edges replicate: the first slice has no predecessor, so its stack is ``(0, 0, 1)``.
    Replication rather than zero padding, because a black channel at the edge tells the
    encoder the anatomy ended, which is false -- the volume ended.
    """
    if n_available < 1:
        raise ValueError(f"n_available must be >= 1, got {n_available}")
    centre = np.asarray(list(centres), dtype=np.int64)
    if centre.size and (centre.min() < 0 or centre.max() >= n_available):
        raise ValueError(f"centre index outside [0, {n_available - 1}]")
    stacked = np.stack([centre - 1, centre, centre + 1], axis=1)
    return np.clip(stacked, 0, n_available - 1)


def _crop_to_fov(volume: Tensor, mm_per_px: float, crop: CropSpec) -> Tensor:
    """Centre-crop ``volume`` (N, C, H, W) to ``crop.fov_mm`` then resize to ``out_px``.

    When the requested field of view is wider than the image -- which happens at the fine
    end of the 0.14 mm/px spacing range -- the crop is zero padded rather than shrunk.
    Shrinking would silently restore the fixed-pixel-crop bug this function removes.
    """
    size = int(volume.shape[-1])
    side = max(1, int(round(crop.fov_mm / mm_per_px)))
    start = (size - side) // 2
    if start < 0:
        pad = -start
        volume = F.pad(volume, (pad, pad, pad, pad))
        start = (int(volume.shape[-1]) - side) // 2
    volume = volume[..., start : start + side, start : start + side]
    if side != crop.out_px:
        volume = F.interpolate(
            volume,
            size=(crop.out_px, crop.out_px),
            mode="bilinear",
            align_corners=False,
            antialias=side > crop.out_px,
        )
    return volume


@dataclass(frozen=True)
class _SeriesRef:
    """Everything needed to load one slot, resolved once at construction."""

    series_uid: str
    path: Path
    spacing: float
    rows: int
    cols: int


class StudyDataset(Dataset):
    """One item per study: up to six slot volumes, twelve labels, two masks.

    ``__getitem__`` returns the per-study half of the pinned batch contract::

        pixels    FloatTensor (n_slots, n_slices, 3, out_px, out_px)
        present   BoolTensor  (n_slots,)   False = the study has no such series
        slot_ids  LongTensor  (n_slots,)   global slot ids, 0..5
        labels    FloatTensor (12,)        all zeros at inference; labels=None
        mask      BoolTensor  (12,)        all False at inference
        study_uid str

    ``slot_ids`` carries the *global* slot index even when ``slots`` selects a subset, so
    a slot embedding trained on the 3-slot cell stays valid in the 6-slot one.

    Duplicate series in one slot are resolved to the one with the most cached slices,
    tie-broken by ``series_uid``, so the choice does not drift between the training run
    and this one.
    """

    def __init__(
        self,
        manifest: pd.DataFrame,
        cache_dir: Path | str,
        *,
        normalisation: Normalisation,
        labels: pd.DataFrame | None = None,
        study_uids: Sequence[str] | None = None,
        slots: Sequence[int] | None = None,
        n_slices: int = 16,
        crop: CropSpec | None = None,
        training: bool = False,
        series_dropout: float = 0.0,
        augment: Callable[[Tensor, np.random.Generator], Tensor] | None = None,
        seed: int = 0,
        verify_cache: bool = True,
        slice_mode: str = "spanning",
    ) -> None:
        for column in (ID_COLUMN_MANIFEST, "series_uid", "slot", "spacing", "rows", "cols"):
            if column not in manifest.columns:
                raise KeyError(f"manifest is missing the {column!r} column")
        if not 0.0 <= series_dropout <= 1.0:
            raise ValueError(f"series_dropout must be in [0, 1], got {series_dropout}")
        if n_slices < 1:
            raise ValueError(f"n_slices must be >= 1, got {n_slices}")
        if slice_mode not in SLICE_SAMPLERS:
            raise ValueError(
                f"unknown slice_mode {slice_mode!r}; known: {sorted(SLICE_SAMPLERS)}"
            )

        self.cache_dir = Path(cache_dir)
        self.crop = crop if crop is not None else CropSpec()
        self.n_slices = int(n_slices)
        self.slice_mode = str(slice_mode)
        self._sample_indices = SLICE_SAMPLERS[self.slice_mode]
        # Series too short for `n_slices`, counted rather than warned per item.
        self._short_series: Counter[str] = Counter()
        self.normalisation = normalisation
        self.training = bool(training)
        self.series_dropout = float(series_dropout)
        self.seed = int(seed)
        self._augment = augment
        self._epoch = 0
        self._mean, self._std = normalisation.tensors(3)

        self._slots = _validate_slots(slots)
        self._slot_ids = torch.tensor(self._slots, dtype=torch.long)
        self._series = self._index_series(manifest, verify_cache=verify_cache)

        if study_uids is None:
            uids = sorted(str(u) for u in manifest[ID_COLUMN_MANIFEST].unique())
        else:
            uids = [str(u) for u in study_uids]
        self._study_uids = tuple(uids)

        self._label_index, self._label_values, self._label_mask = _index_labels(labels)

        # A study with no cached series in the selected slots is emitted as zeros with
        # present=False. At inference that is correct, because every test UID needs a
        # submission row. The warning still fires because the usual cause is a half-built
        # cache.
        self._empty_study_uids = tuple(uid for uid in self._study_uids if not self._series.get(uid))
        if self._empty_study_uids:
            warnings.warn(
                f"{len(self._empty_study_uids)} of {len(self._study_uids)} studies have no "
                f"cached series in slots {self._slots} (e.g. {self._empty_study_uids[0]}); "
                "they are emitted as zeros with present=False everywhere, so they contribute "
                "no signal. Check the cache is complete before reading the score.",
                stacklevel=2,
            )

        if self.training and len(self._slots) > 1 and self.series_dropout == 0.0:
            warnings.warn(
                "training over multiple slots with series_dropout=0.0: absent slots are "
                "zero-filled and flagged, but the encoder never meets an empty slot during "
                "training, so it has no reason to read the presence flag at inference. "
                "Zero-fill, an honest presence flag and ~10% series dropout are one design; "
                "pass series_dropout=0.1.",
                stacklevel=2,
            )

    def set_epoch(self, epoch: int) -> None:
        """Advance the augmentation and dropout stream. Unused at inference."""
        self._epoch = int(epoch)

    @property
    def slots(self) -> tuple[int, ...]:
        """Global slot ids emitted by this dataset, in batch order."""
        return self._slots

    @property
    def study_uids(self) -> tuple[str, ...]:
        return self._study_uids

    @property
    def empty_study_uids(self) -> tuple[str, ...]:
        """Studies with no cached series in the selected slots, i.e. all-absent items."""
        return self._empty_study_uids

    @property
    def short_series(self) -> dict[str, int]:
        """``series_uid -> cached slice count`` for every series shorter than ``n_slices``."""
        return dict(self._short_series)

    def __len__(self) -> int:
        return len(self._study_uids)

    def __getitem__(self, index: int) -> dict[str, Any]:
        uid = self._study_uids[index]
        rng = np.random.default_rng((self.seed, self._epoch, index))
        n_slots = len(self._slots)

        pixels = torch.zeros(
            (n_slots, self.n_slices, 3, self.crop.out_px, self.crop.out_px),
            dtype=torch.float32,
        )
        present = torch.zeros(n_slots, dtype=torch.bool)
        available = self._series.get(uid, {})
        for position, slot in enumerate(self._slots):
            ref = available.get(slot)
            if ref is None:
                continue  # left as exact zeros, and present stays False
            pixels[position] = self._load_slot(ref, rng)
            present[position] = True

        if self.training and self.series_dropout > 0.0:
            self._drop_series(present, pixels, rng)

        labels, mask = self._labels_for(uid)
        if not bool(torch.isfinite(labels).all()):
            raise ValueError(
                f"study {uid} produced non-finite labels {labels.tolist()}; a NaN here "
                "reaches the loss and destroys the run without raising"
            )

        return {
            "pixels": pixels,
            "present": present,
            "slot_ids": self._slot_ids.clone(),
            "labels": labels,
            "mask": mask,
            "study_uid": uid,
        }

    def _index_series(
        self, manifest: pd.DataFrame, *, verify_cache: bool
    ) -> dict[str, dict[int, _SeriesRef]]:
        """Resolve study -> slot -> the one series to load, dropping uncached ones."""
        wanted = set(self._slots)
        index: dict[str, dict[int, _SeriesRef]] = {}
        for row in manifest.itertuples(index=False):
            slot = int(getattr(row, "slot"))
            if slot not in wanted:
                continue
            uid = str(getattr(row, ID_COLUMN_MANIFEST))
            series_uid = str(getattr(row, "series_uid"))
            path = self.cache_dir / uid / f"{series_uid}.npy"
            if verify_cache and not path.is_file():
                continue  # no cache file is genuinely absent, not silently black
            ref = _SeriesRef(
                series_uid=series_uid,
                path=path,
                spacing=float(getattr(row, "spacing")),
                rows=int(getattr(row, "rows")),
                cols=int(getattr(row, "cols")),
            )
            slots = index.setdefault(uid, {})
            incumbent = slots.get(slot)
            if incumbent is None or _prefer(ref, incumbent, manifest_rows=row):
                slots[slot] = ref
        return index

    def _load_slot(self, ref: _SeriesRef, rng: np.random.Generator) -> Tensor:
        volume = np.load(ref.path, mmap_mode="r")
        if volume.ndim != 3 or volume.shape[1] != volume.shape[2]:
            raise ValueError(
                f"{ref.path} has shape {volume.shape}; the cache contract is (S, N, N) uint8"
            )
        n_available = int(volume.shape[0])
        if n_available < self.n_slices:
            self._short_series[ref.series_uid] = n_available
        centres = self._sample_indices(n_available, self.n_slices)
        stack = np.asarray(volume[neighbour_indices(centres, n_available)])

        pixels = torch.from_numpy(np.ascontiguousarray(stack)).to(torch.float32).div_(255.0)
        # Crop in millimetres before normalising: the crop is geometry and the
        # normalisation is the backbone's, and swapping the order would apply the backbone
        # statistics to padding that is not part of the image.
        cache_size = int(volume.shape[-1])
        pixels = _crop_to_fov(
            pixels, cache_mm_per_pixel(ref.spacing, ref.rows, ref.cols, cache_size), self.crop
        )
        pixels = (pixels - self._mean) / self._std
        if self._augment is not None:
            pixels = self._augment(pixels, rng)
            if tuple(pixels.shape[-2:]) != (self.crop.out_px, self.crop.out_px):
                raise ValueError(
                    f"augment hook changed the crop to {tuple(pixels.shape)}; it must "
                    f"preserve ({self.crop.out_px}, {self.crop.out_px})"
                )
        return pixels

    def _drop_series(self, present: Tensor, pixels: Tensor, rng: np.random.Generator) -> None:
        """Zero and un-flag some present slots, in place. Never all of them. Training only."""
        live = [int(i) for i in torch.nonzero(present).flatten().tolist()]
        if len(live) <= 1:
            return  # dropping the only slot leaves the study with no signal at all
        doomed = [i for i in live if rng.random() < self.series_dropout]
        if len(doomed) == len(live):
            doomed.remove(int(rng.choice(np.asarray(doomed))))
        for position in doomed:
            present[position] = False
            pixels[position].zero_()

    def _labels_for(self, uid: str) -> tuple[Tensor, Tensor]:
        row = self._label_index.get(uid)
        if row is None:
            # An unlabelled study is entirely unknown, not entirely negative. This is
            # every study at inference time.
            return torch.zeros(N_LABELS, dtype=torch.float32), torch.zeros(
                N_LABELS, dtype=torch.bool
            )
        return (
            torch.from_numpy(self._label_values[row].copy()),
            torch.from_numpy(self._label_mask[row].copy()),
        )


def _validate_slots(slots: Sequence[int] | None) -> tuple[int, ...]:
    if slots is None:
        return tuple(range(N_SLOTS))
    chosen = tuple(int(s) for s in slots)
    if not chosen:
        raise ValueError("slots must select at least one slot")
    if len(set(chosen)) != len(chosen):
        raise ValueError(f"slots repeats an entry: {chosen}")
    bad = [s for s in chosen if not 0 <= s < N_SLOTS]
    if bad:
        raise ValueError(f"slot ids {bad} outside [0, {N_SLOTS - 1}]")
    return chosen


def _prefer(candidate: _SeriesRef, incumbent: _SeriesRef, *, manifest_rows: Any) -> bool:
    """True when ``candidate`` should replace ``incumbent`` for the same slot.

    More cached slices wins, then the lexicographically smaller series_uid, so the same
    series is chosen here as was chosen during training.
    """
    del manifest_rows  # the tie-break deliberately uses only stable series identity
    candidate_slices = _cached_slice_count(candidate.path)
    incumbent_slices = _cached_slice_count(incumbent.path)
    if candidate_slices != incumbent_slices:
        return candidate_slices > incumbent_slices
    return candidate.series_uid < incumbent.series_uid


def _cached_slice_count(path: Path) -> int:
    """Slice count read from the .npy header, or 0 when the file is unreadable."""
    try:
        return int(np.load(path, mmap_mode="r").shape[0])
    except (OSError, ValueError):
        return 0


def _index_labels(
    labels: pd.DataFrame | None,
) -> tuple[dict[str, int], np.ndarray, np.ndarray]:
    """Flatten a label table into finite values plus a usable-cell mask.

    Always the ``None`` branch here: inference has no labels, and reading report-derived
    labels at test time is a constraint violation, not an optimisation.
    """
    if labels is None:
        return {}, np.zeros((0, N_LABELS), dtype=np.float32), np.zeros((0, N_LABELS), dtype=bool)
    raise ValueError(
        "this notebook is inference-only and must never be handed a label table. "
        "Test studies have no reports and no labels."
    )


def collate_studies(
    samples: Sequence[Mapping[str, Any]],
    *,
    batch_augment: Callable[[dict[str, Any], np.random.Generator], dict[str, Any]] | None = None,
    seed: int = 0,
) -> dict[str, Any]:
    """Stack per-study samples into the pinned batch, re-checking the NaN guarantee."""
    if not samples:
        raise ValueError("cannot collate an empty batch")
    batch: dict[str, Any] = {
        "pixels": torch.stack([s["pixels"] for s in samples]),
        "present": torch.stack([s["present"] for s in samples]),
        "slot_ids": torch.stack([s["slot_ids"] for s in samples]),
        "labels": torch.stack([s["labels"] for s in samples]).to(torch.float32),
        "mask": torch.stack([s["mask"] for s in samples]),
        "study_uid": [str(s["study_uid"]) for s in samples],
    }
    if batch_augment is not None:
        batch = batch_augment(batch, np.random.default_rng(seed))
    if not bool(torch.isfinite(batch["labels"]).all()):
        raise ValueError(
            "batch labels contain a non-finite value; every loss propagates them "
            "silently, so this must never leave the collate"
        )
    return batch

## 7. Model

Copied from `src/kappa/models/{encoder,aggregator,study_classifier}.py`, **verbatim**.

This is the cell where a "cleanup" costs the run. `state_dict` keys are the nn.Module
attribute names and their nesting — `encoder.backbone.stem.0.weight`,
`series_pool.pool.gate.bias`, `study_pool.out_norm.weight`, `head.weight`. Rename one
attribute, reorder one construction, and `load_state_dict(strict=True)` fails; do it in a
way that happens to keep the names and it fails silently instead.

Architecture: one shared 2D backbone over 2.5D three-slice stacks, FiLM-conditioned on the
`plane x contrast` slot (factorised into a plane term and a contrast term, so a
`Sagittal_fs1` gradient also updates the `Sagittal` and `fs1` terms), then two gated
attention pools — slices to a series vector, series to a study vector — and one
`Linear(dim, 12)`.

**The presence mask is load-bearing.** `SliceEncoder` never hands an absent slot to the
backbone at all (`index_select`, not a masked multiply), and `StudyAggregator` substitutes
zeros *before* the slot embedding and the LayerNorm. A model given a zero-filled slot with
no flag learns to read a black volume as anatomy.

**No sigmoid.** Macro ROC-AUC is rank-based, so logits are the output and calibration
cannot change the score.

**Deviation from source:** the volumetric branch of `StudyClassifier.__init__` imports
`VolumeEncoder`, which needs `torchvision`. Both shipped checkpoints are
`convnext_small` (2.5D), so that import is replaced by a raise. Nothing else in the class
changes, and `self.volumetric` is still computed the same way from the backbone name.

In [ ]:
from __future__ import annotations

# ===========================================================================================
# src/kappa/models/encoder.py -- copied verbatim.
# ===========================================================================================

#: Backbones this project supports. Checked against timm.list_pretrained.
RECOMMENDED_BACKBONES: Mapping[str, str] = {
    "convnext_small": "primary; RSNA 2024's winner measured large < base < small",
    "tf_efficientnetv2_s": "diversity member; BatchNorm backbone, different error modes",
    "caformer_s18": "diversity member; hybrid conv/attention",
    "maxvit_tiny_tf_384": "diversity member; native 384 px, and mean/std of 0.5 not ImageNet",
}

#: The 3D backbones, which are torchvision.models.video names and NOT timm names. Naming
#: one of these in model.backbone is the entire switch onto VolumeEncoder.
VIDEO_BACKBONES: tuple[str, ...] = (
    "r3d_18",
    "mc3_18",
    "r2plus1d_18",
)

#: Name suffixes that mark a scaled-up variant of a supported family. Warned about rather
#: than refused, because the decision belongs to whoever reads the warning.
_SCALED_UP_SUFFIXES = ("_base", "_large", "_huge", "_giant", "_xl", "_l", "_m", "_b36", "_m36")

#: Standard deviation of the FiLM conditioning at initialisation. Overwritten by the
#: checkpoint; kept so the construction order matches the trained model exactly.
_FILM_INIT_STD = 0.02


@dataclass(frozen=True)
class BackboneInfo:
    """What the encoder actually loaded, as opposed to what was asked for.

    ``img_size_applied`` is *stored*, not recovered from ``input_size``: input_size comes
    from timm's pretrained_cfg, which describes the checkpoint rather than the object that
    was built.
    """

    name: str
    pretrained_requested: bool
    weights_loaded: bool
    pretrained_tag: str | None
    hf_hub_id: str | None
    num_features: int
    input_size: tuple[int, int, int]
    mean: tuple[float, ...]
    std: tuple[float, ...]
    drop_path_applied: float | None
    n_batchnorm: int
    img_size_requested: int | None = None
    img_size_applied: int | None = None

    def render(self) -> str:
        """Multi-line rendering for the top of a run log."""
        weights = "pretrained" if self.weights_loaded else "RANDOM INIT (pretrained=False)"
        if self.weights_loaded:
            source = self.hf_hub_id or self.pretrained_tag or "unknown"
        else:
            source = "n/a"
        drop_path = "not supported by this family" if self.drop_path_applied is None else (
            f"{self.drop_path_applied:g}"
        )
        return "\n".join(
            [
                f"backbone   : {self.name}  weights={weights}  source={source}",
                f"features   : dim={self.num_features}  batchnorm_layers={self.n_batchnorm}",
                f"preprocess : input_size={self.input_size}  mean={self.mean}  std={self.std}"
                "   <- from timm, never hardcoded",
                f"img_size   : {self._render_img_size()}",
                f"drop_path  : {drop_path}",
            ]
        )

    def _render_img_size(self) -> str:
        """The one line that can contradict input_size on the line above it."""
        native = self.input_size[-1]
        if self.img_size_requested is None:
            return f"not requested; built at the family's native {native}"
        if self.img_size_applied is None:
            return (
                f"{self.img_size_requested} REQUESTED BUT DROPPED -- this family does not "
                "accept img_size, so the model was built at its native geometry"
            )
        if self.img_size_applied != native:
            return (
                f"{self.img_size_applied}  <- built at this size; position embeddings "
                f"interpolated from the native {native}, which is the size input_size "
                "above still reports"
            )
        return f"{self.img_size_applied} (the family's native size)"


def backbone_data_config(model: nn.Module) -> dict[str, Any]:
    """Resolve the backbone's own preprocessing config. Local; touches no network."""
    return dict(timm.data.resolve_data_config(model=model))


def create_backbone(
    name: str = "convnext_small",
    *,
    pretrained: bool = True,
    drop_path_rate: float = 0.2,
    img_size: int | None = None,
) -> tuple[nn.Module, BackboneInfo]:
    """Build a pooled-feature timm backbone and report exactly what came back.

    ``pretrained=False`` is what this notebook always passes, and with it timm never
    resolves a hub id, so nothing is fetched -- which is the only reason this runs offline.

    ``img_size`` is a "do not pass the argument at all" when None, and the distinction is
    load-bearing: for a convolutional backbone like convnext_small the family REFUSES the
    argument, and the enumeration below drops it and records that it was dropped.
    """
    if name.endswith(_SCALED_UP_SUFFIXES):
        warnings.warn(
            f"backbone {name!r} looks like a scaled-up variant. Scaling the encoder is a "
            "measured null on this dataset: DINOv2-S to DINOv2-B was worth +0.0011 against "
            "a 0.0020 noise floor, improving 5 of 12 labels. Scale the input geometry "
            f"instead. Supported: {sorted(RECOMMENDED_BACKBONES)}.",
            stacklevel=2,
        )
    if pretrained and not timm.list_pretrained(name):
        hint = ""
        if timm.list_pretrained(f"tf_{name}"):
            hint = f" Did you mean {'tf_' + name!r}?"
        raise ValueError(
            f"backbone {name!r} has no pretrained weights in timm, so pretrained=True would "
            f"return a randomly initialised model without failing.{hint} "
            f"Supported: {sorted(RECOMMENDED_BACKBONES)}."
        )

    kwargs: dict[str, Any] = {"pretrained": pretrained, "num_classes": 0, "global_pool": "avg"}

    # The optional kwargs, i.e. the ones some timm families refuse. Each is tried and, if
    # refused, dropped and recorded -- building without them is fine, silently building
    # without them is not.
    optional: dict[str, Any] = {"drop_path_rate": drop_path_rate}
    if img_size is not None:
        optional["img_size"] = int(img_size)

    # Retried with one kwarg removed at a time, not with the whole set removed on the
    # first TypeError. A family that refused only drop_path_rate would otherwise lose the
    # img_size it does honour, and a ViT would hold a 518-px position grid while the
    # loader hands it 392.
    attempts: list[dict[str, Any]] = [optional]
    if len(optional) > 1:
        attempts += [{k: v for k, v in optional.items() if k != drop} for drop in optional]

    # Head-norm placement. global_pool="avg" makes timm build fc_norm and leave norm as
    # Identity. DINOv2's published weights were trained with token pooling and ship norm.*,
    # so the module tree and the checkpoint disagree and load_state_dict raises inside
    # create_model. fc_norm=False restores the checkpoint's norm and keeps mean pooling.
    # Tried only after the default fails. RuntimeError only: a TypeError means the family
    # has no such argument at all, which the enumeration above already handles.
    def _build(attempt: dict[str, Any]) -> nn.Module:
        try:
            return timm.create_model(name, **attempt, **kwargs)
        except RuntimeError as exc:
            if "fc_norm" not in str(exc):
                raise
            rebuilt = timm.create_model(name, **attempt, **kwargs, fc_norm=False)
            warnings.warn(
                f"backbone {name!r} ships a token-pooling checkpoint whose head norm is "
                f"'norm', which does not match the 'fc_norm' that global_pool='avg' "
                f"builds. Rebuilt with fc_norm=False, so the pretrained norm IS loaded "
                f"and pooling stays 'avg', comparable with the other backbones.",
                stacklevel=2,
            )
            return rebuilt

    model: nn.Module | None = None
    used: dict[str, Any] = {}
    for attempt in attempts:
        try:
            model = _build(attempt)
        except TypeError:
            continue
        used = attempt
        break
    if model is None:
        # Unguarded on purpose. If the bare call raises TypeError too then nothing here was
        # ever about these kwargs.
        model = _build({})

    applied: float | None = drop_path_rate if "drop_path_rate" in used else None
    config = backbone_data_config(model)
    pretrained_cfg = dict(getattr(model, "pretrained_cfg", {}) or {})
    info = BackboneInfo(
        name=name,
        pretrained_requested=pretrained,
        weights_loaded=bool(pretrained),
        pretrained_tag=pretrained_cfg.get("tag") or None,
        hf_hub_id=pretrained_cfg.get("hf_hub_id") or None,
        num_features=int(model.num_features),
        input_size=tuple(int(v) for v in config["input_size"]),  # type: ignore[arg-type]
        mean=tuple(float(v) for v in config["mean"]),
        std=tuple(float(v) for v in config["std"]),
        drop_path_applied=applied,
        n_batchnorm=sum(
            1 for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)
        ),
        img_size_requested=None if img_size is None else int(img_size),
        img_size_applied=used.get("img_size"),
    )
    return model, info


def _slot_tables() -> tuple[Tensor, Tensor]:
    """Plane index and contrast bit for each slot, derived from SLOT_ORDER."""
    planes = torch.tensor([PLANES.index(plane) for plane, _ in SLOT_ORDER], dtype=torch.long)
    contrast = torch.tensor([int(fs) for _, fs in SLOT_ORDER], dtype=torch.long)
    return planes, contrast


class SliceEncoder(nn.Module):
    """Shared 2D backbone over 2.5D three-slice stacks, conditioned on the slot.

    The three input channels are *adjacent slices*, not colour, which is why the
    backbone's own normalisation must be applied identically to all three.

    Conditioning is FiLM on the pooled embedding, factorised into a plane term and a
    contrast term rather than a single six-way lookup.
    """

    def __init__(
        self,
        backbone: str = "convnext_small",
        *,
        pretrained: bool = True,
        drop_path_rate: float = 0.2,
        n_slots: int = N_SLOTS,
        img_size: int | None = None,
    ) -> None:
        super().__init__()
        if not 1 <= n_slots <= len(SLOT_ORDER):
            # A job may select fewer slots, but slot_ids stay global, so the conditioning
            # tables can never be shorter than the ids they are indexed by.
            raise ValueError(f"n_slots={n_slots} outside 1..{len(SLOT_ORDER)}")
        self.backbone, self.info = create_backbone(
            backbone,
            pretrained=pretrained,
            drop_path_rate=drop_path_rate,
            img_size=img_size,
        )
        self.n_slots = int(n_slots)
        self.embed_dim = self.info.num_features

        planes, contrast = _slot_tables()
        if self.n_slots != len(SLOT_ORDER):
            planes, contrast = planes[: self.n_slots], contrast[: self.n_slots]
        # Buffers rather than plain tensors so `.to(device)` moves them with the model.
        # persistent=False, so they are not in the state dict -- which is why a strict
        # load does not complain about them.
        self.register_buffer("slot_plane", planes, persistent=False)
        self.register_buffer("slot_contrast", contrast, persistent=False)

        self.plane_film = nn.Embedding(len(PLANES), 2 * self.embed_dim)
        self.contrast_film = nn.Embedding(2, 2 * self.embed_dim)
        nn.init.normal_(self.plane_film.weight, std=_FILM_INIT_STD)
        nn.init.normal_(self.contrast_film.weight, std=_FILM_INIT_STD)

    @property
    def data_config(self) -> dict[str, Any]:
        """The backbone's own preprocessing config. Not used at inference -- the
        normalisation comes out of the checkpoint, never re-resolved."""
        return backbone_data_config(self.backbone)

    def describe(self) -> str:
        """What was actually loaded, for the run log."""
        return self.info.render()

    def set_grad_checkpointing(self, enable: bool = True) -> None:
        """Trade compute for activation memory in the backbone. Unused at inference."""
        setter = getattr(self.backbone, "set_grad_checkpointing", None)
        if setter is None:
            warnings.warn(
                f"backbone {self.info.name!r} does not support gradient checkpointing; "
                "activation memory is unchanged.",
                stacklevel=2,
            )
            return
        setter(enable=enable)

    def forward(self, pixels: Tensor, slot_ids: Tensor, present: Tensor | None = None) -> Tensor:
        """Embed every slice stack of every present slot.

        Returns ``(B, S, N, embed_dim)``, exactly zero on absent slots.
        """
        if pixels.ndim != 6:
            raise ValueError(
                f"pixels has shape {tuple(pixels.shape)}; the pinned batch is "
                "(B, N_SLOTS, n_slices, 3, H, W)"
            )
        batch, slots, n_slices, channels, height, width = pixels.shape
        if channels != 3:
            raise ValueError(
                f"pixels has {channels} channels; the 2.5D neighbour stack is pinned at 3"
            )
        if tuple(slot_ids.shape) != (batch, slots):
            raise ValueError(
                f"slot_ids has shape {tuple(slot_ids.shape)}, expected {(batch, slots)}"
            )
        if present is None:
            present = pixels.new_ones((batch, slots), dtype=torch.bool)
        elif tuple(present.shape) != (batch, slots):
            raise ValueError(
                f"present has shape {tuple(present.shape)}, expected {(batch, slots)}"
            )
        present = present.bool()

        flat_present = present.unsqueeze(-1).expand(batch, slots, n_slices).reshape(-1)
        flat_pixels = pixels.reshape(-1, channels, height, width)
        flat_slots = slot_ids.unsqueeze(-1).expand(batch, slots, n_slices).reshape(-1)

        out = flat_pixels.new_zeros((flat_pixels.shape[0], self.embed_dim))
        keep = flat_present.nonzero(as_tuple=True)[0]
        if keep.numel():
            # index_select, not a boolean mask, so the gather is a real gather: the absent
            # rows are not merely multiplied by zero downstream, they are never handed to
            # the backbone at all.
            features = self.backbone(flat_pixels.index_select(0, keep))
            features = self._condition(features, flat_slots.index_select(0, keep))
            out = out.index_copy(0, keep, features.to(out.dtype))
        return out.view(batch, slots, n_slices, self.embed_dim)

    def _condition(self, features: Tensor, slot_ids: Tensor) -> Tensor:
        """Apply factorised plane x contrast FiLM to pooled features."""
        if int(slot_ids.max()) >= self.n_slots or int(slot_ids.min()) < 0:
            raise ValueError(
                f"slot_ids outside [0, {self.n_slots - 1}]: saw "
                f"[{int(slot_ids.min())}, {int(slot_ids.max())}]"
            )
        plane = self.plane_film(self.slot_plane[slot_ids])
        contrast = self.contrast_film(self.slot_contrast[slot_ids])
        gamma, beta = (plane + contrast).chunk(2, dim=-1)
        return features * (1.0 + gamma) + beta

In [ ]:
from __future__ import annotations

# ===========================================================================================
# src/kappa/models/aggregator.py -- copied verbatim.
# ===========================================================================================


def _head_count(dim: int, preferred: int = 4) -> int:
    """Largest head count not exceeding ``preferred`` that divides ``dim``.

    Embedding width is whatever the backbone reports -- 768 for convnext_small. A fixed
    head count would make a legal backbone raise deep inside nn.MultiheadAttention.
    """
    for heads in (preferred, 8, 4, 2, 1):
        if heads <= preferred and dim % heads == 0:
            return heads
    return 1


class GatedAttentionPool(nn.Module):
    """Gated attention pooling over a sequence, with an optional exclusion mask.

    Gated rather than plain attention because a single tanh branch cannot suppress an
    element: it can only weight it. The sigmoid gate multiplies the score down toward zero.
    """

    def __init__(self, dim: int, hidden: int | None = None) -> None:
        super().__init__()
        hidden = hidden or max(32, dim // 4)
        self.value = nn.Linear(dim, hidden)
        self.gate = nn.Linear(dim, hidden)
        self.score = nn.Linear(hidden, 1)

    def forward(self, x: Tensor, mask: Tensor | None = None) -> tuple[Tensor, Tensor]:
        """Pool ``(B, T, D)`` to ``(B, D)``, returning ``(pooled, weights (B, T))``.

        ``mask`` is True where the element participates. Excluded elements are substituted
        with zeros before any linear layer sees them, so a non-finite value in an excluded
        position cannot reach the output or the gradient.
        """
        if x.ndim != 3:
            raise ValueError(f"expected (B, T, D), got {tuple(x.shape)}")
        batch, length, _ = x.shape
        if mask is None:
            mask = x.new_ones((batch, length), dtype=torch.bool)
        elif tuple(mask.shape) != (batch, length):
            raise ValueError(f"mask has shape {tuple(mask.shape)}, expected {(batch, length)}")
        mask = mask.bool()

        keep = mask.unsqueeze(-1)
        x = torch.where(keep, x, torch.zeros_like(x))

        logits = self.score(torch.tanh(self.value(x)) * torch.sigmoid(self.gate(x))).squeeze(-1)
        # finfo.min rather than -inf: under autocast this tensor may be fp16, where -inf
        # survives the softmax but poisons any later arithmetic on the logits.
        excluded = torch.full_like(logits, torch.finfo(logits.dtype).min)
        logits = torch.where(mask, logits, excluded)
        # A row with nothing present would softmax a row of identical minima. That is
        # uniform, not NaN, but it would still average in the zeros; zero the weights so
        # the pooled vector is exactly zero and the caller sees an honest nothing.
        empty = ~mask.any(dim=-1, keepdim=True)
        logits = torch.where(empty, torch.zeros_like(logits), logits)
        weights = logits.softmax(dim=-1)
        weights = torch.where(empty, torch.zeros_like(weights), weights)
        return (weights.unsqueeze(-1) * x).sum(dim=1), weights


class SeriesAggregator(nn.Module):
    """Ordered slice embeddings to one series vector.

    A transformer over the slice axis, then gated attention pooling. Order is supplied by
    a learned positional table because slice position carries anatomy: the corpus is
    11.7x anisotropic with a median 3.00 mm slice thickness.
    """

    def __init__(
        self,
        dim: int,
        n_slices: int = 16,
        n_layers: int = 2,
        n_heads: int = 4,
        mlp_ratio: float = 2.0,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        if dim <= 0 or n_slices <= 0:
            raise ValueError(f"dim={dim} and n_slices={n_slices} must both be positive")
        self.dim = int(dim)
        self.n_slices = int(n_slices)
        layer = nn.TransformerEncoderLayer(
            d_model=self.dim,
            nhead=_head_count(self.dim, n_heads),
            dim_feedforward=int(self.dim * mlp_ratio),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers, enable_nested_tensor=False)
        self.position = nn.Parameter(torch.zeros(1, self.n_slices, self.dim))
        nn.init.trunc_normal_(self.position, std=0.02)
        self.norm = nn.LayerNorm(self.dim)
        self.pool = GatedAttentionPool(self.dim)

    def positional(self, length: int) -> Tensor:
        """Positional embedding for a sequence of ``length`` slices.

        Interpolated when ``length`` differs from the configured n_slices. Slices are
        sampled to span the volume, so index i of length sits at a *fraction* of the way
        through the knee. The equal-length path returns the table untouched, so the common
        case is exact.
        """
        if length == self.n_slices:
            return self.position
        table = self.position.transpose(1, 2)
        return F.interpolate(table, size=length, mode="linear", align_corners=False).transpose(1, 2)

    def forward(self, slices: Tensor) -> Tensor:
        """Reduce ``(B, T, D)`` slice embeddings to ``(B, D)`` series vectors."""
        if slices.ndim != 3:
            raise ValueError(f"expected (B, n_slices, dim), got {tuple(slices.shape)}")
        if slices.shape[-1] != self.dim:
            raise ValueError(f"slice embedding is {slices.shape[-1]}-dim, expected {self.dim}")
        hidden = self.encoder(slices + self.positional(slices.shape[1]))
        pooled, _ = self.pool(self.norm(hidden))
        return pooled


class StudyAggregator(nn.Module):
    """Series vectors plus a slot embedding to one study vector, honouring presence.

    The slot embedding is added rather than concatenated so the pool sees one width
    regardless of how many slots a job selects -- which is exactly why the 3-slot and
    6-slot checkpoints share this module unchanged. ``present`` is required, not optional.
    """

    def __init__(self, dim: int, n_slots: int = 6, dropout: float = 0.0) -> None:
        super().__init__()
        if dim <= 0 or n_slots <= 0:
            raise ValueError(f"dim={dim} and n_slots={n_slots} must both be positive")
        self.dim = int(dim)
        self.n_slots = int(n_slots)
        self.slot_embedding = nn.Embedding(self.n_slots, self.dim)
        nn.init.normal_(self.slot_embedding.weight, std=0.02)
        self.norm = nn.LayerNorm(self.dim)
        self.dropout = nn.Dropout(dropout)
        self.pool = GatedAttentionPool(self.dim)
        self.out_norm = nn.LayerNorm(self.dim)

    def forward(
        self, series: Tensor, present: Tensor, slot_ids: Tensor
    ) -> tuple[Tensor, Tensor]:
        """Fuse ``(B, S, D)`` series vectors into ``(B, D)``, returning the weights.

        Absent slots are substituted with zeros *first*, before the slot embedding, the
        LayerNorm or the attention scores. Doing it in that order is what makes an absent
        slot unable to change the result even when it holds nan.
        """
        if series.ndim != 3:
            raise ValueError(f"expected (B, n_slots, dim), got {tuple(series.shape)}")
        batch, slots, dim = series.shape
        if dim != self.dim:
            raise ValueError(f"series vector is {dim}-dim, expected {self.dim}")
        if tuple(present.shape) != (batch, slots):
            raise ValueError(
                f"present has shape {tuple(present.shape)}, expected {(batch, slots)}"
            )
        if tuple(slot_ids.shape) != (batch, slots):
            raise ValueError(
                f"slot_ids has shape {tuple(slot_ids.shape)}, expected {(batch, slots)}"
            )
        present = present.bool()

        keep = present.unsqueeze(-1)
        series = torch.where(keep, series, torch.zeros_like(series))
        hidden = self.norm(series + self.slot_embedding(slot_ids))
        hidden = torch.where(keep, hidden, torch.zeros_like(hidden))
        pooled, weights = self.pool(self.dropout(hidden), present)
        return self.out_norm(pooled), weights


class LabelQueryHead(nn.Module):
    """One learned query per label, cross-attending the study's slice tokens.

    An ablation arm, not a default. Copied so that StudyClassifier's construction is
    byte-for-byte the source's; neither shipped checkpoint uses it (both carry
    ``head.weight`` of shape (12, 768), i.e. head="linear").
    """

    def __init__(
        self,
        dim: int,
        n_labels: int,
        num_heads: int = 8,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.dim = int(dim)
        self.n_labels = int(n_labels)
        self.num_heads = _head_count(dim, preferred=num_heads)
        self.queries = nn.Parameter(torch.zeros(self.n_labels, self.dim))
        self.attention = nn.MultiheadAttention(
            self.dim, self.num_heads, dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(self.dim)
        # A per-label projection to a scalar, rather than a shared Linear: a shared one
        # would put the twelve back on a common readout and undo the point.
        self.weight = nn.Parameter(torch.zeros(self.n_labels, self.dim))
        self.bias = nn.Parameter(torch.zeros(self.n_labels))
        nn.init.trunc_normal_(self.queries, std=0.02)
        nn.init.trunc_normal_(self.weight, std=0.01)

    def forward(self, tokens: Tensor, token_mask: Tensor) -> Tensor:
        """Score ``(B, T, D)`` tokens into ``(B, n_labels)`` logits."""
        if tokens.ndim != 3:
            raise ValueError(f"expected (B, T, D), got {tuple(tokens.shape)}")
        batch, length, dim = tokens.shape
        if dim != self.dim:
            raise ValueError(f"tokens are {dim}-dim, expected {self.dim}")
        if tuple(token_mask.shape) != (batch, length):
            raise ValueError(
                f"token_mask has shape {tuple(token_mask.shape)}, expected {(batch, length)}"
            )
        token_mask = token_mask.bool()

        # Zero excluded tokens before the projections inside the attention.
        tokens = torch.where(token_mask.unsqueeze(-1), tokens, torch.zeros_like(tokens))

        # A row with no tokens at all would make key_padding_mask all-True and
        # nn.MultiheadAttention return NaN. Attend uniformly instead, then zero the result.
        empty = ~token_mask.any(dim=-1)
        keys_valid = token_mask | empty.unsqueeze(-1)

        queries = self.queries.unsqueeze(0).expand(batch, -1, -1)
        attended, _ = self.attention(
            queries, tokens, tokens, key_padding_mask=~keys_valid, need_weights=False
        )
        attended = torch.where(empty.view(batch, 1, 1), torch.zeros_like(attended), attended)
        attended = self.norm(attended)
        return (attended * self.weight).sum(dim=-1) + self.bias

    def extra_repr(self) -> str:
        return f"dim={self.dim}, n_labels={self.n_labels}, num_heads={self.num_heads}"

In [ ]:
from __future__ import annotations

# ===========================================================================================
# src/kappa/models/study_classifier.py -- copied verbatim except the VolumeEncoder import
# (see the markdown above).
# ===========================================================================================


@dataclass(frozen=True)
class ModelOutput:
    """Logits for the 12 scored labels, plus the optional auxiliary head.

    Logits, not probabilities: macro ROC-AUC is rank-based and threshold-free.
    """

    logits: Tensor
    aux_logits: Tensor | None = None
    study_vector: Tensor | None = None


class StudyClassifier(nn.Module):
    """One shared encoder, two attention pools, twelve masked logits.

    Takes the batch exactly as collate_studies emits it -- pixels
    (B, N_SLOTS, n_slices, 3, H, W), present (B, N_SLOTS), slot_ids (B, N_SLOTS).
    """

    def __init__(
        self,
        backbone: str = "convnext_small",
        n_slices: int = 16,
        pretrained: bool = True,
        drop_path_rate: float = 0.2,
        aux_dim: int = 0,
        head: str = "linear",
        head_num_heads: int = 8,
        freeze_backbone_bn: bool = False,
        img_size: int | None = None,
    ) -> None:
        super().__init__()
        if aux_dim < 0:
            raise ValueError(f"aux_dim={aux_dim} must be non-negative")
        if head not in ("linear", "query"):
            raise ValueError(f"unknown head {head!r}; known: 'linear', 'query'")
        # The backbone name is the whole switch between the 2.5D and volumetric arms.
        self.volumetric = backbone.split(".", 1)[0] in VIDEO_BACKBONES
        if self.volumetric:
            # Deviation from source: upstream imports kappa.models.volume_encoder here,
            # which needs torchvision. Neither shipped checkpoint is volumetric, so this
            # refuses by name instead of inlining an encoder that would never run.
            raise NotImplementedError(
                f"backbone {backbone!r} is a volumetric (torchvision video) arm. This "
                "notebook ships only the 2.5D encoder; both bundled checkpoints are "
                "convnext_small. Inline kappa.models.volume_encoder.VolumeEncoder to "
                "score a 3D cell."
            )
        else:
            self.encoder = SliceEncoder(
                backbone,
                pretrained=pretrained,
                drop_path_rate=drop_path_rate,
                img_size=img_size,
            )
        dim = self.encoder.embed_dim
        self.n_slices = int(n_slices)
        self.aux_dim = int(aux_dim)
        self.series_pool = SeriesAggregator(
            dim=dim, n_slices=1 if self.volumetric else self.n_slices
        )
        self.study_pool = StudyAggregator(dim=dim, n_slots=N_SLOTS)
        self.head_kind = str(head)
        if self.head_kind == "query":
            self.head = LabelQueryHead(dim=dim, n_labels=N_LABELS, num_heads=head_num_heads)
        else:
            self.head = nn.Linear(dim, N_LABELS)
        self.aux_head = nn.Linear(dim, self.aux_dim) if self.aux_dim else None

        # Small weights and a zero bias on the fresh linear heads. Overwritten by the
        # checkpoint immediately; kept so that construction matches the trained model and
        # nothing is left uninitialised if a key is ever missing.
        linear_heads = [self.aux_head]
        if self.head_kind != "query":
            linear_heads.append(self.head)
        for linear in linear_heads:
            if linear is not None:
                nn.init.trunc_normal_(linear.weight, std=0.01)
                nn.init.zeros_(linear.bias)

        print(self.describe())

    @property
    def data_config(self) -> dict[str, Any]:
        """The backbone's own preprocessing config. NOT used here -- the normalisation
        comes out of the checkpoint payload, which is the only thing guaranteed to match
        the weights."""
        return self.encoder.data_config

    @property
    def n_parameters(self) -> int:
        """Total parameter count, for the run log and for memory budgeting."""
        return sum(p.numel() for p in self.parameters())

    def describe(self) -> str:
        """Everything a reader needs to know what this run is evidence about."""
        aux = "disabled" if not self.aux_dim else f"{self.aux_dim} report-supervised units"
        if self.head_kind == "query":
            shape = (
                f"{N_LABELS} learned queries x {self.head.num_heads} attn heads over slice tokens"
            )
        else:
            shape = f"one shared Linear({self.encoder.embed_dim}, {N_LABELS})"
        return "\n".join(
            [
                "StudyClassifier",
                self.encoder.describe(),
                (
                    f"pooling    : {self.n_slices}-slice VOLUME -> 1 token/series -> "
                    f"{N_SLOTS} slots (presence-masked) -> study"
                    if self.volumetric
                    else f"pooling    : {self.n_slices} slices/series -> series -> "
                    f"{N_SLOTS} slots (presence-masked) -> study"
                ),
                f"head       : {self.head_kind} -- {shape}",
                f"heads      : {N_LABELS} labels ({', '.join(LABELS[:3])}, ...)  aux={aux}",
                f"parameters : {self.n_parameters:,}",
            ]
        )

    def set_grad_checkpointing(self, enable: bool = True) -> None:
        """Trade backbone compute for activation memory. Unused at inference."""
        self.encoder.set_grad_checkpointing(enable)

    def forward(self, pixels: Tensor, present: Tensor, slot_ids: Tensor) -> ModelOutput:
        """Score one batch of studies. Returns ModelOutput with logits (B, 12)."""
        embeddings = self.encoder(pixels, slot_ids, present)
        batch, slots, n_slices, dim = embeddings.shape
        # One flat call rather than a loop over slots: the series pool never attends across
        # slots, so folding them into the batch axis is exact.
        series = self.series_pool(embeddings.reshape(batch * slots, n_slices, dim))
        study, _ = self.study_pool(series.view(batch, slots, dim), present, slot_ids)
        aux = self.aux_head(study) if self.aux_head is not None else None
        if self.head_kind == "query":
            tokens = embeddings.reshape(batch, slots * n_slices, dim)
            token_mask = (
                present.bool().unsqueeze(-1).expand(batch, slots, n_slices).reshape(batch, -1)
            )
            return ModelOutput(
                logits=self.head(tokens, token_mask), aux_logits=aux, study_vector=study
            )
        return ModelOutput(logits=self.head(study), aux_logits=aux, study_vector=study)

## 8. Checkpoint loader

Copied from `src/kappa/inference/predict.py`. The checkpoint is **self-describing**:
backbone, slots, `n_slices`, crop and normalisation all come out of the payload, so no YAML
and no repo is needed.

Two lines carry the whole offline story:

* **`pretrained=False`, hardcoded and unconditional.** With it, `timm.create_model` never
  resolves a hub id and never touches the network. Asking for published weights here would
  fail *after* the architecture was built, so the traceback would blame the backbone rather
  than the missing network.
* **`img_size` comes from `payload["crop"]["out_px"]`**, which *is* the input matrix — the
  dataset resizes every crop to exactly that. The wrong `img_size` loads every weight
  successfully, reports nothing wrong, and then asserts on the first forward pass, inside
  the notebook, with the budget already spent. (For `convnext_small` the family refuses the
  argument and `create_backbone` records that it dropped it — which is correct, and the
  same thing that happened at train time.)

A teacher checkpoint is refused by name rather than by a wall of unexpected keys: it reads
report features, and test studies have no reports.

In [ ]:
from __future__ import annotations

# ===========================================================================================
# src/kappa/inference/predict.py -- checkpoint half.
# ===========================================================================================

#: Bumped whenever the checkpoint payload changes shape. A checkpoint written by an older
#: version is refused by name rather than silently missing a key, because the key most
#: likely to go missing is `normalisation` and its absence is invisible.
CHECKPOINT_VERSION = 1


@dataclass(frozen=True)
class LoadedCheckpoint:
    """A model restored from disk, together with the preprocessing it was trained with.

    The preprocessing is the point. Building a dataset from anything other than
    ``normalisation`` and ``crop`` gives the backbone pixels it was not trained on, which
    still produces a number -- so ``dataset()`` takes no normalisation argument at all.
    """

    model: StudyClassifier
    normalisation: Normalisation
    crop: CropSpec
    slots: tuple[int, ...]
    n_slices: int
    meta: dict[str, Any]
    path: Path

    @property
    def backbone(self) -> str:
        return str(self.meta.get("backbone", "unknown"))

    @property
    def slice_mode(self) -> str:
        """Which slice sampler this checkpoint was TRAINED with.

        Read from ``meta`` rather than a top-level key, so every file written before the
        volumetric arm existed is a "spanning" run -- which is what the default says.
        Scoring a contiguous-trained model with the spanning sampler would feed it a
        full-extent spread it never saw and still print an AUC.
        """
        return str(self.meta.get("slice_mode", "spanning"))

    def describe(self) -> str:
        """What this checkpoint is evidence about. Print it before scoring anything."""
        return "\n".join(
            [
                f"checkpoint : {self.path}",
                f"backbone   : {self.backbone}  (weights from the checkpoint, "
                f"pretrained=False so nothing is fetched)",
                f"trained as : config {self.meta.get('config_hash')} seed "
                f"{self.meta.get('seed')} fold {self.meta.get('fold')} "
                f"epoch {self.meta.get('epoch')} sha {self.meta.get('git_sha')}",
                f"sampling   : {self.slice_mode} over {self.n_slices} slices",
                f"input      : slots={list(self.slots)} n_slices={self.n_slices} "
                f"{self.crop.fov_mm:.0f} mm / {self.crop.out_px} px = "
                f"{self.crop.mm_per_pixel:.4f} mm/px",
                f"normalise  : mean={self.normalisation.mean} std={self.normalisation.std} "
                f"(from the checkpoint, never re-resolved)",
                f"fold macro : {self.meta.get('macro')}",
            ]
        )

    def dataset(
        self,
        manifest: pd.DataFrame,
        cache_dir: Path | str,
        *,
        study_uids: Sequence[str] | None = None,
        verify_cache: bool = True,
    ) -> StudyDataset:
        """Build an inference dataset with this checkpoint's own geometry.

        ``training=False`` and ``series_dropout=0.0`` are not arguments: dropping a series
        at inference would throw away signal for no benefit, and augmentation at inference
        is a different model from the one that was validated.
        """
        return StudyDataset(
            manifest,
            cache_dir,
            normalisation=self.normalisation,
            labels=None,
            study_uids=study_uids,
            slots=self.slots,
            n_slices=self.n_slices,
            crop=self.crop,
            training=False,
            series_dropout=0.0,
            verify_cache=verify_cache,
            slice_mode=self.slice_mode,
        )


def load_checkpoint(
    path: Path | str,
    *,
    device: str | torch.device | None = None,
    strict: bool = True,
) -> LoadedCheckpoint:
    """Rebuild the model and its preprocessing from a checkpoint, touching no network."""
    path = Path(path)
    resolved = resolve_device(device)
    # weights_only=True refuses to unpickle arbitrary objects. This payload is tensors and
    # plain containers by construction, so the safe loader is sufficient.
    try:
        payload = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        # torch < 1.13 has no weights_only. These checkpoints are the project's own files,
        # so this fallback is for compatibility only.
        payload = torch.load(path, map_location="cpu")
    if not isinstance(payload, dict):
        raise ValueError(f"{path} does not hold a KAPPA checkpoint mapping")
    # A teacher reads report text, which no test study has. Its state dict carries a report
    # branch and a fusion layer this architecture has no slot for, so a strict load would
    # fail here anyway -- but with a wall of unexpected-key names that reads like a version
    # mismatch. Say the real thing instead.
    if payload.get("kappa_teacher"):
        raise ValueError(
            f"{path} is a teacher checkpoint and cannot be used for inference. The "
            "teacher reads report features, which test studies do not have. Submit "
            "the distilled student instead."
        )

    version = payload.get("kappa_checkpoint_version")
    if version != CHECKPOINT_VERSION:
        raise ValueError(
            f"{path} has checkpoint version {version!r}, this build reads "
            f"{CHECKPOINT_VERSION}. Refusing rather than guessing: the key most likely "
            "to be missing is 'normalisation', and a wrong mean/std still produces "
            "scores."
        )
    for key in ("state_dict", "normalisation", "crop", "slots", "n_slices"):
        if key not in payload:
            raise ValueError(f"{path} is missing the {key!r} key")

    meta = dict(payload.get("meta") or {})
    model = StudyClassifier(
        backbone=str(meta.get("backbone", "convnext_small")),
        n_slices=int(payload["n_slices"]),
        pretrained=False,
        drop_path_rate=float(meta.get("drop_path_rate", 0.0)),
        aux_dim=int(meta.get("aux_dim", 0)),
        # `meta` does not record `head`, so the default "linear" is correct -- both
        # shipped cells carry head.weight of shape (12, 768). A mismatch would fail
        # loudly on the strict load below.
        img_size=int(payload["crop"]["out_px"]),
    )
    model.load_state_dict(payload["state_dict"], strict=strict)
    model.eval().to(resolved)

    return LoadedCheckpoint(
        model=model,
        normalisation=Normalisation(
            tuple(float(v) for v in payload["normalisation"]["mean"]),
            tuple(float(v) for v in payload["normalisation"]["std"]),
        ),
        crop=CropSpec(
            fov_mm=float(payload["crop"]["fov_mm"]), out_px=int(payload["crop"]["out_px"])
        ),
        slots=tuple(int(s) for s in payload["slots"]),
        n_slices=int(payload["n_slices"]),
        meta=meta,
        path=path,
    )

## 9. Test-time augmentation and the prediction pass

Copied from `src/kappa/inference/predict.py` and `src/kappa/data/augment.py`.

**There is no flip.** A horizontal flip swaps medial and lateral
on a knee. Medial/Lateral Meniscus and Medial/Lateral OA could be swapped back, but MCL has
no LCL column and PF OA is not lateralised, so no relabelling makes a flipped study correct
for all twelve labels. It is a wrong-label augmentation that raises nothing and quietly
costs AUC.

**TTA pools by mean of logits, not by rank** — the one place this project's
"rank-average, never probability-average" rule does not apply. Rank normalisation is
*transductive*: a study's rank depends on which other studies were in the same call, so
rank-pooling views would make one study's score a function of its batch-mates. Logit
averaging is per-study and local.

`TTA = ()` takes a separate code branch that is bit-identical to the pre-TTA arithmetic.

In [ ]:
from __future__ import annotations

# ===========================================================================================
# src/kappa/inference/predict.py + src/kappa/data/augment.py::rotate -- copied verbatim.
# ===========================================================================================

#: Every test-time augmentation this notebook will run, and the complete legal set.
TTA_TRANSFORMS: tuple[str, ...] = (
    "identity",
    "rotate_cw",
    "rotate_ccw",
    "zoom_in",
    "zoom_out",
)

#: Half-angle of the rotate_cw / rotate_ccw pair, in degrees. 10 because that is the
#: train-time augment.rotate_deg: a TTA view drawn from the distribution the model was
#: trained over is a view it has seen. The pair is symmetric about 0 on purpose.
TTA_ROTATE_DEGREES = 10.0

#: Linear magnification of zoom_in; zoom_out uses its reciprocal. 1.1 is deliberately
#: small: there is NO train-time scale jitter, so a zoom view is outside the training
#: distribution by construction.
TTA_ZOOM_FACTOR = 1.1


def rotate(pixels: Tensor, degrees: float) -> Tensor:
    """Rotate ``(..., H, W)`` about its centre, preserving the shape exactly.

    The crop is a physical measurement (fov_mm / out_px mm/px), so a transform that resized
    it would silently change the scale. Corners vacated by the rotation are filled with
    0.0, which downstream of the backbone's normalisation is the mean pixel value -- and
    notably not black, which would be a real intensity.
    """
    if degrees == 0.0:
        return pixels
    lead, height, width = pixels.shape[:-2], pixels.shape[-2], pixels.shape[-1]
    flat = pixels.reshape(-1, 1, height, width)
    theta = torch.tensor(float(np.deg2rad(degrees)), dtype=pixels.dtype)
    cos, sin = torch.cos(theta), torch.sin(theta)
    # affine_grid maps output coords to input coords, so this is the inverse rotation.
    matrix = torch.tensor(
        [[cos, -sin, 0.0], [sin, cos, 0.0]], dtype=pixels.dtype, device=pixels.device
    ).expand(flat.shape[0], 2, 3)
    grid = F.affine_grid(matrix, list(flat.shape), align_corners=False)
    turned = F.grid_sample(
        flat, grid, mode="bilinear", padding_mode="zeros", align_corners=False
    )
    return turned.reshape(*lead, height, width)


def _zoom(pixels: torch.Tensor, factor: float) -> torch.Tensor:
    """Scale ``(..., H, W)`` about its centre by ``factor``, onto the *same* pixel grid.

    ``factor > 1`` magnifies. What moves is the apparent scale of the anatomy within a
    fixed grid; CropSpec.mm_per_pixel still describes the grid that comes back.
    """
    if factor <= 0.0:
        raise ValueError(f"zoom factor must be positive, got {factor}")
    if factor == 1.0:
        return pixels
    lead, height, width = pixels.shape[:-2], pixels.shape[-2], pixels.shape[-1]
    flat = pixels.reshape(-1, 1, height, width)
    # affine_grid maps output coords to input coords, so the matrix is the *inverse* of
    # the magnification.
    inverse = 1.0 / float(factor)
    matrix = torch.tensor(
        [[inverse, 0.0, 0.0], [0.0, inverse, 0.0]], dtype=pixels.dtype, device=pixels.device
    ).expand(flat.shape[0], 2, 3)
    grid = F.affine_grid(matrix, list(flat.shape), align_corners=False)
    scaled = F.grid_sample(
        flat, grid, mode="bilinear", padding_mode="zeros", align_corners=False
    )
    return scaled.reshape(*lead, height, width)


def _tta_views(names: Sequence[str]) -> tuple[Callable[[torch.Tensor], torch.Tensor], ...]:
    """Resolve TTA names to deterministic, shape-preserving pixel operations.

    Deterministic is load-bearing: a random TTA would give one checkpoint a different macro
    AUC on every evaluation, and the gap between working and top-10 on this task is about
    0.006.
    """
    resolved = tuple(str(name) for name in names)
    unknown = [name for name in resolved if name not in TTA_TRANSFORMS]
    if unknown:
        raise ValueError(
            f"unknown tta {unknown}; legal names are {list(TTA_TRANSFORMS)}. There is no "
            "flip in that set and adding one would mislabel every lateralised column."
        )
    if len(set(resolved)) != len(resolved):
        # A repeat is not a harmless no-op: it double-weights one view in the mean and pays
        # a full extra forward pass for it.
        raise ValueError(f"duplicate tta transform in {list(resolved)}")
    if not resolved:
        return ()

    # Positive degrees turns the image anticlockwise on screen under `rotate`'s
    # output-to-input convention, so `rotate_cw` takes the negative angle.
    table: dict[str, Callable[[torch.Tensor], torch.Tensor]] = {
        "identity": lambda pixels: pixels,
        "rotate_cw": lambda pixels: rotate(pixels, -TTA_ROTATE_DEGREES),
        "rotate_ccw": lambda pixels: rotate(pixels, TTA_ROTATE_DEGREES),
        "zoom_in": lambda pixels: _zoom(pixels, TTA_ZOOM_FACTOR),
        "zoom_out": lambda pixels: _zoom(pixels, 1.0 / TTA_ZOOM_FACTOR),
    }
    return tuple(table[name] for name in resolved)


def _autocast_dtype(device: torch.device, amp: bool | str | None) -> torch.dtype | None:
    """Resolve the autocast dtype, or None for a full-precision pass."""
    if amp is False or amp == "off":
        return None
    if amp is None or amp is True:
        return torch.bfloat16 if device.type == "cuda" else None
    return {"bf16": torch.bfloat16, "fp16": torch.float16}[str(amp)]


@dataclass(frozen=True)
class PredictionResult:
    """Scores for a set of studies, with the wall clock that produced them.

    ``scores`` are raw logits in LABELS order -- no sigmoid, because the metric ranks
    within a column and a sigmoid is monotone. ``seconds`` is measured around the whole
    pass including the dataloader, which is the number the notebook limit applies to.
    """

    study_uids: tuple[str, ...]
    scores: np.ndarray
    seconds: float
    n_batches: int
    device: str
    amp: str
    tta: tuple[str, ...] = ()

    def __post_init__(self) -> None:
        if self.scores.shape != (len(self.study_uids), N_LABELS):
            raise ValueError(
                f"scores {self.scores.shape} do not match {len(self.study_uids)} studies "
                f"x {N_LABELS} labels"
            )
        if not np.isfinite(self.scores).all():
            raise ValueError(
                "prediction contains a non-finite score. A NaN cannot be ranked and a "
                "submission carrying one is rejected outright, so this fails here rather "
                "than at upload."
            )

    @property
    def n_studies(self) -> int:
        return len(self.study_uids)

    @property
    def n_passes(self) -> int:
        """Forward passes per study. 1 with no TTA, len(tta) with it."""
        return max(len(self.tta), 1)

    @property
    def n_forward_batches(self) -> int:
        """Batches the *model* saw, i.e. loader batches times TTA views."""
        return self.n_batches * self.n_passes

    @property
    def seconds_per_study(self) -> float:
        """Measured, not estimated. Already inclusive of every TTA view."""
        return self.seconds / max(self.n_studies, 1)

    def projection(self, n_studies: int) -> tuple[float, float]:
        """``(projected seconds, runtime term)`` for a full test set of ``n_studies``."""
        seconds = self.seconds_per_study * float(n_studies)
        return seconds, seconds / NOTEBOOK_LIMIT_SECONDS

    def timing_report(self, n_studies: int | None = None) -> str:
        """One block stating what was measured and what was only projected from it."""
        lines = [
            f"inference  : {self.seconds:.2f} s for {self.n_studies} studies "
            f"({self.n_batches} batches) on {self.device}, amp={self.amp}",
            f"           : {self.seconds_per_study:.4f} s/study measured",
        ]
        if self.tta:
            lines.append(
                f"tta        : {self.n_passes} view(s) {list(self.tta)}, averaged in "
                "logit space (never by rank -- see the markdown above)"
            )
            lines.append(
                f"           : {self.n_forward_batches} forward batches; the rate above "
                "already includes all of them"
            )
        if n_studies:
            seconds, term = self.projection(n_studies)
            lines.append(
                f"projection : {seconds:.1f} s for {n_studies} studies = "
                f"{100.0 * seconds / NOTEBOOK_LIMIT_SECONDS:.1f}% of the "
                f"{NOTEBOOK_LIMIT_SECONDS} s notebook limit, runtime term {term:.4f}"
            )
        return "\n".join(lines)

    def frame(self) -> pd.DataFrame:
        """StudyInstanceUID plus the twelve score columns, in submission order."""
        data: dict[str, Any] = {ID_COLUMN: list(self.study_uids)}
        for position, name in enumerate(LABELS):
            data[name] = self.scores[:, position]
        return pd.DataFrame(data)


def _forward_logits(
    model: StudyClassifier,
    pixels: torch.Tensor,
    present: torch.Tensor,
    slot_ids: torch.Tensor,
    *,
    device_type: str,
    dtype: torch.dtype | None,
) -> torch.Tensor:
    """One forward pass, returned as float32 logits still on the device.

    float32 before anything else happens to them: a bf16 logit carries about three decimal
    digits, ROC-AUC reads ties, and keeping reduced precision would manufacture ties
    between studies the model actually separated.
    """
    if dtype is None:
        out = model(pixels, present, slot_ids)
    else:
        with torch.autocast(device_type=device_type, dtype=dtype):
            out = model(pixels, present, slot_ids)
    return out.logits.detach().float()


@torch.no_grad()
def predict_studies(
    model: StudyClassifier,
    dataset: StudyDataset,
    *,
    device: str | torch.device | None = None,
    batch_size: int = 8,
    num_workers: int = 0,
    pin_memory: bool | None = None,
    amp: bool | str | None = None,
    tta: Sequence[str] = (),
) -> PredictionResult:
    """Score every study in ``dataset`` and measure how long the whole pass took.

    The timer starts before the DataLoader is iterated and stops after the last batch, so
    decoding, cropping and collation are inside it. Timing only the forward call
    underestimated wall clock by 1.7x three times running on this project.

    The loader is iterated **once** whichever branch runs: decoding and cropping dominate a
    CPU pass, and re-reading the cache per view would multiply the part of the cost that
    TTA has no reason to pay.
    """
    views = _tta_views(tta)
    resolved = resolve_device(device)
    dtype = _autocast_dtype(resolved, amp)
    amp_label = "off" if dtype is None else f"{resolved.type}/{str(dtype).split('.')[-1]}"
    was_training = model.training
    model.eval().to(resolved)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=bool(pin_memory) if pin_memory is not None else resolved.type == "cuda",
        collate_fn=collate_studies,
        drop_last=False,
    )

    uids: list[str] = []
    chunks: list[np.ndarray] = []
    n_batches = 0
    started = time.perf_counter()
    for batch in loader:
        pixels = batch["pixels"].to(resolved, non_blocking=True)
        present = batch["present"].to(resolved, non_blocking=True)
        slot_ids = batch["slot_ids"].to(resolved, non_blocking=True)
        if not views:
            # Kept as its own branch rather than folded into the loop below with an
            # identity view. The default has to be the pre-TTA arithmetic exactly.
            logits = _forward_logits(
                model, pixels, present, slot_ids, device_type=resolved.type, dtype=dtype
            )
        else:
            # The view runs outside autocast, on the float32 pixels the loader produced.
            # Resampling at bf16's ~3 decimal digits would blur the geometry the view
            # exists to perturb.
            total = None
            for view in views:
                seen = _forward_logits(
                    model,
                    view(pixels),
                    present,
                    slot_ids,
                    device_type=resolved.type,
                    dtype=dtype,
                )
                total = seen if total is None else total + seen
            # Mean of logits, not of ranks. Dividing by 1.0 is exact, so a single-view tta
            # gives bit-identical scores to no tta at all.
            logits = total / float(len(views))
        chunks.append(logits.cpu().numpy())
        uids.extend(batch["study_uid"])
        n_batches += 1
    seconds = time.perf_counter() - started

    if was_training:
        model.train()
    scores = (
        np.concatenate(chunks, axis=0) if chunks else np.zeros((0, N_LABELS), dtype=np.float32)
    )
    return PredictionResult(
        study_uids=tuple(uids),
        scores=np.ascontiguousarray(scores, dtype=np.float64),
        seconds=seconds,
        n_batches=n_batches,
        device=str(resolved),
        amp=amp_label,
        tta=tuple(str(name) for name in tta),
    )

## 10. Submission

Copied from `src/kappa/inference/submission.py` and `src/kappa/evaluation/metrics.py`.

* The header comes from `SUBMISSION_COLUMNS` **and nowhere else**, is written with `csv` at
  `QUOTE_MINIMAL`, and is read back off disk and compared byte for byte before the writer
  returns. A BOM, a stray quote or CRLF is invisible on the object you passed in.
* The line terminator is `\n`, not `csv.writer`'s default `\r\n`. A known-good constant-0.5
  file for the three public test rows is **470 bytes** = 130 header + 1 + 3 x (64-byte UID +
  48 bytes of `,0.5` + 1). CRLF makes the same file 474.
* **Blending is per-column rank average, never a probability average.** A probability mean
  is dominated by whichever member happens to be more spread out — influence bought with
  confidence, which this competition never scores. Macro ROC-AUC is rank-invariant, so
  calibration is irrelevant by construction.

In [ ]:
from __future__ import annotations

# ===========================================================================================
# src/kappa/evaluation/metrics.py -- the rank transform.
# ===========================================================================================


def average_ranks(values: np.ndarray) -> np.ndarray:
    """One-based ranks of a 1-D array, ties sharing their mid-rank.

    Tie handling is where a hand-rolled AUC silently diverges from sklearn, and one
    implementation cannot disagree with itself.
    """
    values = np.asarray(values, dtype=np.float64).ravel()
    n = values.size
    if n == 0:
        return np.empty(0, dtype=np.float64)
    order = np.argsort(values, kind="stable")
    ordered = values[order]
    is_new = np.empty(n, dtype=bool)
    is_new[0] = True
    np.not_equal(ordered[1:], ordered[:-1], out=is_new[1:])
    starts = np.flatnonzero(is_new)
    counts = np.diff(np.append(starts, n))
    # A tie group starting at index s with c members spans ranks s+1 .. s+c, whose mean is
    # s + (c + 1) / 2.
    mid = starts + (counts + 1.0) / 2.0
    ranks = np.empty(n, dtype=np.float64)
    ranks[order] = np.repeat(mid, counts)
    return ranks


def rank_normalise(scores: np.ndarray) -> np.ndarray:
    """Map each column independently onto (0, 1) by its mid-ranks.

    Independently *per column* is the load-bearing word: the twelve labels are twelve
    separate rankings, so a shared normalisation would mix twelve unrelated orderings.
    Within a column the map is monotone, so it leaves that column's AUC bit-identical --
    which is exactly why it is safe to apply before combining models.
    """
    scores = np.asarray(scores, dtype=np.float64)
    if scores.ndim != 2:
        raise ValueError(f"rank_normalise expects (n_studies, n_labels), got {scores.shape}")
    if np.isnan(scores).any():
        raise ValueError("rank_normalise got NaN; a NaN score has no rank")
    out = np.empty_like(scores)
    n = scores.shape[0]
    for column in range(scores.shape[1]):
        out[:, column] = (average_ranks(scores[:, column]) - 0.5) / max(n, 1)
    return out


# ===========================================================================================
# src/kappa/inference/submission.py -- copied verbatim.
# ===========================================================================================

#: Newline used on every row. See the markdown above for the byte arithmetic.
LINE_TERMINATOR = "\n"

#: One identifier column plus twelve labels. A row with any other field count is malformed,
#: and an unquoted apostrophe or space is exactly how that happens.
N_FIELDS = len(SUBMISSION_COLUMNS)


def expected_header_bytes() -> bytes:
    """The header line, encoded, derived from SUBMISSION_COLUMNS alone.

    Built through csv rather than ",".join so that the expectation and the write share one
    quoting implementation.
    """
    buffer = io.StringIO()
    writer = csv.writer(buffer, quoting=csv.QUOTE_MINIMAL, lineterminator=LINE_TERMINATOR)
    writer.writerow(SUBMISSION_COLUMNS)
    return buffer.getvalue().encode("utf-8")


def _as_scores(scores) -> np.ndarray:
    """Coerce predictions to a finite (N, 12) float array, or explain why not."""
    array = np.asarray(scores, dtype=np.float64)
    if array.ndim == 1:
        array = array.reshape(1, -1)
    if array.ndim != 2 or array.shape[1] != N_LABELS:
        raise ValueError(
            f"scores must be (N, {N_LABELS}) to match the twelve labels, got {array.shape}"
        )
    if not np.isfinite(array).all():
        # NaN in a submission is not a degraded score, it is a rejected file.
        bad = np.argwhere(~np.isfinite(array))
        first = tuple(bad[0])
        raise ValueError(
            f"{len(bad)} non-finite score(s); first at row {first[0]}, "
            f"label {LABELS[first[1]]!r}. Every study needs a finite score for all "
            f"{N_LABELS} labels -- masking applies to the loss, never to the output."
        )
    return array


def write_submission(
    path: str | Path,
    study_uids: Sequence[str],
    scores,
    *,
    float_format: str = "%.6f",
) -> Path:
    """Write submission.csv and verify its header byte for byte.

    The header check re-reads the file rather than trusting the writer, because the failure
    mode being guarded against -- a BOM, a quoted header, CRLF from a different writer --
    is invisible on the object that was passed in.
    """
    path = Path(path)
    array = _as_scores(scores)
    uids = [str(uid) for uid in study_uids]
    if len(uids) != len(array):
        raise ValueError(f"{len(uids)} study UIDs but {len(array)} score rows")
    if len(set(uids)) != len(uids):
        raise ValueError("duplicate StudyInstanceUID; the leaderboard joins on this column")

    path.parent.mkdir(parents=True, exist_ok=True)
    # newline="" is required: without it csv's terminator is translated again by the text
    # layer on some platforms and the byte count stops being stable.
    with open(path, "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, quoting=csv.QUOTE_MINIMAL, lineterminator=LINE_TERMINATOR)
        writer.writerow(SUBMISSION_COLUMNS)
        for uid, row in zip(uids, array):
            writer.writerow([uid, *(float_format % value for value in row)])

    with open(path, "rb") as handle:
        written = handle.readline()
    expected = expected_header_bytes()
    if written != expected:
        raise AssertionError(
            "submission header does not match SUBMISSION_COLUMNS byte for byte:\n"
            f"  wrote    {written!r}\n  expected {expected!r}"
        )
    return path


def read_submission(path: str | Path) -> tuple[list[str], list[str], np.ndarray]:
    """Return ``(header, uids, scores)`` exactly as the file states them."""
    path = Path(path)
    with open(path, "r", encoding="utf-8", newline="") as handle:
        rows = list(csv.reader(handle))
    if not rows:
        raise ValueError(f"{path} is empty")
    header, *body = rows
    uids = [row[0] for row in body if row]
    values = np.array(
        [[float(cell) for cell in row[1:]] for row in body if row and len(row) == N_FIELDS],
        dtype=np.float64,
    ) if body else np.zeros((0, N_LABELS))
    return header, uids, values


@dataclass(frozen=True)
class SubmissionCheck:
    """Outcome of the seven contract checks, one entry each.

    A check whose reference data was not supplied reports None -- not checked -- and is
    counted separately, because a validator that scores an unrun check as a pass is worse
    than no validator.
    """

    path: Path
    results: dict[str, bool | None]
    notes: dict[str, str]
    n_rows: int
    n_bytes: int

    @property
    def ok(self) -> bool:
        """True only if no check failed. Unchecked entries do not count as passes."""
        return all(value is not False for value in self.results.values())

    @property
    def n_passed(self) -> int:
        return sum(1 for value in self.results.values() if value is True)

    @property
    def n_checks(self) -> int:
        return len(self.results)

    def summary(self) -> str:
        lines = [f"{self.path}  {self.n_bytes} bytes, {self.n_rows} rows"]
        for name, value in self.results.items():
            mark = {True: "PASS", False: "FAIL", None: "----"}[value]
            note = self.notes.get(name, "")
            lines.append(f"  [{mark}] {name}" + (f" -- {note}" if note else ""))
        lines.append(f"  {self.n_passed}/{self.n_checks} checks passed")
        return "\n".join(lines)


def validate_submission(
    path: str | Path, *, expected_uids: Sequence[str] | None = None
) -> SubmissionCheck:
    """Run the seven submission contract checks against a file on disk.

    Header exact, apostrophe intact, row count, UID order, field count, values finite, UIDs
    unique. ``expected_uids`` comes from test.csv; without it the two order-and-count checks
    report as not checked rather than as passes.
    """
    path = Path(path)
    raw = path.read_bytes()
    header, uids, values = read_submission(path)
    results: dict[str, bool | None] = {}
    notes: dict[str, str] = {}

    header_line = raw.split(b"\n", 1)[0] + b"\n"
    results["header exact"] = header_line == expected_header_bytes()
    if not results["header exact"]:
        notes["header exact"] = f"{header_line!r}"

    results["Baker's apostrophe intact"] = "Baker's" in header
    if not results["Baker's apostrophe intact"]:
        notes["Baker's apostrophe intact"] = f"header field 10 is {header[10:11]}"

    if expected_uids is None:
        results["row count matches test.csv"] = None
        results["UID order matches test.csv"] = None
        notes["row count matches test.csv"] = "no expected_uids supplied"
        notes["UID order matches test.csv"] = "no expected_uids supplied"
    else:
        wanted = [str(uid) for uid in expected_uids]
        results["row count matches test.csv"] = len(uids) == len(wanted)
        notes["row count matches test.csv"] = f"{len(uids)} rows vs {len(wanted)} expected"
        results["UID order matches test.csv"] = uids == wanted

    with open(path, "r", encoding="utf-8", newline="") as handle:
        widths = {len(row) for row in csv.reader(handle) if row}
    results["13 fields per row"] = widths == {N_FIELDS}
    notes["13 fields per row"] = f"field counts seen: {sorted(widths)}"

    results["values finite"] = bool(values.size) and bool(np.isfinite(values).all())
    results["UIDs unique"] = len(set(uids)) == len(uids)

    return SubmissionCheck(
        path=path, results=results, notes=notes, n_rows=len(uids), n_bytes=len(raw)
    )


def rank_blend(score_sets: Iterable[np.ndarray]) -> np.ndarray:
    """Combine several models' predictions by per-column rank normalisation.

    The only supported ensemble. Averaging probabilities lets the more confident model
    dominate a metric that only reads order, and calibration cannot move a rank-based
    metric at all.
    """
    stack = [rank_normalise(np.asarray(scores, dtype=np.float64)) for scores in score_sets]
    if not stack:
        raise ValueError("rank_blend needs at least one set of scores")
    shapes = {array.shape for array in stack}
    if len(shapes) != 1:
        raise ValueError(f"score sets disagree on shape: {sorted(shapes)}")
    return np.mean(stack, axis=0)


def transform_scores(scores: np.ndarray, transform: str = "rank") -> np.ndarray:
    """Prepare logits for writing. "rank" is per-column rank normalisation.

    Both options give a bit-identical macro ROC-AUC, because both are monotone within a
    column and the metric reads nothing else. "rank" is the default only because it lands
    in [0, 1].
    """
    scores = np.asarray(scores, dtype=np.float64)
    if transform == "none":
        return scores
    if transform != "rank":
        raise ValueError(f"unknown transform {transform!r}; use 'rank' or 'none'")
    if scores.shape[0] == 0:
        return scores
    return rank_normalise(scores)


def write_predictions(
    path: Path | str,
    results: Sequence[PredictionResult],
    *,
    transform: str = "rank",
    expected_uids: Sequence[str] | None = None,
) -> Path:
    """Blend one or more members by rank and write submission.csv.

    ``expected_uids`` reorders the rows to test.csv's order. The leaderboard joins on
    StudyInstanceUID, but a row-order mismatch is free to fix here and expensive to
    discover later.
    """
    if not results:
        raise ValueError("write_predictions needs at least one PredictionResult")
    uid_lists = [result.study_uids for result in results]
    if len({tuple(u) for u in uid_lists}) != 1:
        raise ValueError(
            "members disagree on their study order; rank_blend combines by position, so "
            "a differing order would blend one study's rank with another's"
        )

    uids = list(uid_lists[0])
    if len(results) == 1:
        blended = transform_scores(results[0].scores, transform)
    else:
        # rank_blend already rank-normalises each member, so the blend is in [0, 1]
        # whatever `transform` says; applying it again would be a monotone no-op.
        blended = rank_blend([result.scores for result in results])

    if expected_uids is not None:
        wanted = [str(u) for u in expected_uids]
        position = {uid: index for index, uid in enumerate(uids)}
        missing = [uid for uid in wanted if uid not in position]
        if missing:
            raise ValueError(
                f"{len(missing)} expected study UID(s) were not scored, e.g. {missing[:3]}. "
                "Every test study needs a row: a short submission scores as an error."
            )
        order = [position[uid] for uid in wanted]
        blended = blended[order]
        uids = wanted

    return write_submission(path, uids, blended)

## 11. Resolve the mounts

The competition mount is `/kaggle/input/competitions/rsna-knee-abnormality-detection/` —
**not** the classic `/kaggle/input/<slug>/`, which does not exist for this competition.
Both are tried, then a bounded search for any directory holding `test_series.csv`, and
the error lists what was actually found.

`test.csv` is a **(3, 1) placeholder** in the public copy; the real test set is swapped in
at rescore. **Nothing below sizes an array, a batch count or a loop bound from those three
rows.** Every loop is driven by what is actually mounted.

In [ ]:
from __future__ import annotations


def _dir_listing(root: str, depth: int = 2) -> list[str]:
    """A short human listing of what is under a mount, for an error message."""
    base = Path(root)
    if not base.is_dir():
        return [f"{base}  (does not exist)"]
    out = []
    for child in sorted(base.iterdir())[:40]:
        out.append(str(child) + ("/" if child.is_dir() else ""))
        if depth > 1 and child.is_dir():
            for grand in sorted(child.iterdir())[:10]:
                out.append("    " + str(grand) + ("/" if grand.is_dir() else ""))
    return out


def resolve_comp_dir(explicit=None, series_csv_name: str = "test_series.csv") -> Path:
    """Locate the directory holding test_series.csv."""
    if explicit is not None:
        candidate = Path(explicit)
        if (candidate / series_csv_name).is_file():
            return candidate
        raise FileNotFoundError(
            f"COMP_DIR={candidate} does not hold {series_csv_name}. Contents:\n  "
            + "\n  ".join(_dir_listing(str(candidate)))
        )
    for name in COMP_DIR_CANDIDATES:
        candidate = Path(name)
        if (candidate / series_csv_name).is_file():
            return candidate
    # Bounded search. glob("**") over /kaggle/input can be slow on a large attached
    # dataset, so this walks explicit depths instead.
    root = Path("/kaggle/input")
    if root.is_dir():
        for pattern in (f"*/{series_csv_name}", f"*/*/{series_csv_name}",
                        f"*/*/*/{series_csv_name}"):
            found = sorted(root.glob(pattern))
            if found:
                return found[0].parent
    raise FileNotFoundError(
        f"could not find {series_csv_name}. The documented mount is\n  "
        f"{COMP_DIR_CANDIDATES[0]}\n"
        "and the classic /kaggle/input/<slug>/ path does NOT exist for this competition.\n"
        "Set COMP_DIR explicitly in the Config cell. What is actually under /kaggle/input:\n  "
        + "\n  ".join(_dir_listing("/kaggle/input"))
    )


def resolve_series_root(comp_dir: Path, explicit=None, stem: str = "test_series") -> Path:
    """Locate <study_uid>/<series_uid>/*.dcm for the test split."""
    if explicit is not None:
        candidate = Path(explicit)
        if candidate.is_dir():
            return candidate
        raise NotADirectoryError(f"SERIES_ROOT={candidate} is not a directory")
    # The train split is <root>/train_series.csv beside <root>/train_series/, so the test
    # split is assumed to mirror it.
    for name in (stem, "test_images", "test"):
        candidate = comp_dir / name
        if candidate.is_dir():
            return candidate
    raise NotADirectoryError(
        f"no series directory under {comp_dir}; tried {stem}/, test_images/, test/. "
        f"Set SERIES_ROOT explicitly. Contents:\n  " + "\n  ".join(_dir_listing(str(comp_dir), 1))
    )


def _walk_dirs(root, max_depth: int = 9):
    """Directories under root, shallowest first, bounded and pruned.

    Pruned because /kaggle/input also holds the competition mount, and this
    competition is 819,640 files -- an unpruned walk there costs minutes for nothing.
    """
    base = Path(root)
    if not base.is_dir():
        return
    n = len(base.parts)
    found = []
    for dirpath, dirnames, _ in os.walk(base):
        here = Path(dirpath)
        depth = len(here.parts) - n
        if depth >= max_depth:
            dirnames[:] = []
        # Never descend into the competition data or any DICOM series tree.
        dirnames[:] = [d for d in dirnames
                       if d not in ("competitions", "test_series", "train_series")]
        found.append((depth, here))
    for _, d in sorted(found, key=lambda t: t[0]):
        yield d


def resolve_weights_dir(explicit=None, cells=("v2_fs_three_planes_convnext",)) -> Path:
    """Locate the attached dataset holding the checkpoint cell directories."""
    if explicit is not None:
        candidate = Path(explicit)
        if candidate.is_dir():
            return candidate
        raise NotADirectoryError(f"WEIGHTS_DIR={candidate} is not a directory")
    def holds_cells(d: Path) -> bool:
        return all((d / c).is_dir() for c in cells)

    def holds_flat(d: Path) -> bool:
        # The Kaggle CLI defaults to --dir-mode skip, so a flat upload is the likeliest
        # accident; supporting it costs nothing.
        return any(d.glob("*fold*_seed*.pt"))

    # 1. Declared candidate globs, in order, exact shapes first.
    for pattern in WEIGHTS_DIR_CANDIDATES:
        for candidate in sorted(Path("/").glob(pattern.lstrip("/"))):
            if candidate.is_dir() and (holds_cells(candidate) or holds_flat(candidate)):
                return candidate
    # 2. Depth-bounded, pruned walk, shallowest first.
    for candidate in _walk_dirs("/kaggle/input"):
        if holds_cells(candidate) or holds_flat(candidate):
            return candidate
    raise FileNotFoundError(
        "could not find the checkpoint dataset. Expected a directory containing "
        f"{list(cells)}. Attach it and/or set WEIGHTS_DIR in the Config cell. "
        "What is under /kaggle/input:\n  " + "\n  ".join(_dir_listing("/kaggle/input"))
    )


T0 = time.perf_counter()

COMP_DIR = resolve_comp_dir(COMP_DIR)
SERIES_CSV = COMP_DIR / "test_series.csv"
TEST_CSV = COMP_DIR / "test.csv"
SERIES_ROOT = resolve_series_root(COMP_DIR, SERIES_ROOT)
WEIGHTS_DIR = resolve_weights_dir(
    WEIGHTS_DIR, cells=tuple(sorted({c.split("/")[0] for c in CHECKPOINTS}))
)

print(f"comp dir    : {COMP_DIR}")
print(f"series csv  : {SERIES_CSV}   ({SERIES_CSV.stat().st_size} bytes)")
print(f"series root : {SERIES_ROOT}")
print(f"weights dir : {WEIGHTS_DIR}")
print(f"cache dir   : {CACHE_DIR}")
print(f"out path    : {OUT_PATH}")

In [ ]:
from __future__ import annotations

# --- The study list that drives every loop below ------------------------------------------
# Read from test.csv when it exists, because that is what defines the required submission
# rows and their order. It is a (3, 1) placeholder in the public copy and the real one is
# swapped in at rescore, so its length is never assumed -- only its contents are used.
if TEST_CSV.is_file():
    _test = pd.read_csv(TEST_CSV)
    if ID_COLUMN in _test.columns:
        TEST_UIDS = [str(u) for u in _test[ID_COLUMN]]
    else:
        # A one-column placeholder whose header is spelled differently must not silently
        # produce an empty submission.
        warnings.warn(
            f"{TEST_CSV} has no {ID_COLUMN!r} column (columns: {list(_test.columns)}); "
            f"using the first column {_test.columns[0]!r} as the study identifier.",
            stacklevel=2,
        )
        TEST_UIDS = [str(u) for u in _test[_test.columns[0]]]
else:
    warnings.warn(
        f"{TEST_CSV} does not exist; falling back to the unique studies in "
        f"{SERIES_CSV.name}. Row ORDER then comes from the series index, not from "
        "test.csv, which is fine because the leaderboard joins on the UID column.",
        stacklevel=2,
    )
    TEST_UIDS = sorted(read_series_index(SERIES_CSV)["study_uid"].unique().tolist())

if len(set(TEST_UIDS)) != len(TEST_UIDS):
    raise ValueError("test.csv repeats a StudyInstanceUID; the submission joins on it")

if LIMIT is not None:
    # Smoke test only. The resulting submission is short, and validate_submission
    # reports it.
    TEST_UIDS = sorted(TEST_UIDS)[: int(LIMIT)]
    print(f"!! LIMIT={LIMIT} -- scoring {len(TEST_UIDS)} studies. THIS IS NOT A SUBMISSION.")

N_TEST = len(TEST_UIDS)
print(f"test studies: {N_TEST}")
print(f"first three : {TEST_UIDS[:3]}")

# Write a placeholder submission.csv as soon as the study list is known, before any
# decoding or model loading. Later cells overwrite it, so a failure anywhere after this
# point still leaves a scoreable file (0.5 everywhere, i.e. AUC 0.5) instead of an error.
if WRITE_PARTIAL_SUBMISSION:
    write_submission(OUT_PATH, TEST_UIDS, np.full((N_TEST, len(LABELS)), 0.5))
    print(f"\nfallback    : {OUT_PATH} ({N_TEST} rows of 0.5) -- placeholder, overwritten below")

## 12. Manifest and slice cache

One header-only DICOM read per series to fill `spacing`, `rows`, `cols`, then a full decode
of every series into `CACHE_DIR/<study_uid>/<series_uid>.npy`.

The cache build is the dominant cost and is **resumable** — every file is written to a
temporary name and `os.replace`d into place, so a restarted notebook picks up where it
stopped and never reads a truncated array.

A study that has rows in `test_series.csv` but no directory on disk is skipped from the
manifest and still gets a submission row (all slots absent). That is deliberate: a missing
UID fails the submission contract outright, and an honest all-absent study is the correct
answer for a study with no usable pixels.

In [ ]:
from __future__ import annotations

_t = time.perf_counter()
MANIFEST = build_manifest(
    SERIES_CSV,
    SERIES_ROOT,
    studies=TEST_UIDS,
    require_all=False,
    progress_every=200,
)
MANIFEST_SECONDS = time.perf_counter() - _t

print(MANIFEST.head())
print(
    f"\nmanifest    : {len(MANIFEST)} series over "
    f"{MANIFEST['study_uid'].nunique()} studies in {MANIFEST_SECONDS:.1f}s"
)
print("slot counts :")
print(MANIFEST["slot"].value_counts().sort_index().to_string())

# A study in test.csv with no manifest rows has no series directory on disk. It still gets
# a row in the submission -- StudyDataset emits it as all-absent -- but it is worth seeing.
_no_series = sorted(set(TEST_UIDS) - set(MANIFEST["study_uid"]))
if _no_series:
    print(
        f"\n!! {len(_no_series)} of {N_TEST} test studies have NO series on disk "
        f"(e.g. {_no_series[:3]}). They will be scored as all-slots-absent and still "
        "get a submission row, because a missing UID fails the contract."
    )

# spacing is what converts 150 mm into a pixel crop. A null here would raise inside
# cache_mm_per_pixel, so surface it now rather than at the first batch.
_bad_geom = MANIFEST[
    MANIFEST["spacing"].isna() | (MANIFEST["rows"] < 1) | (MANIFEST["cols"] < 1)
]
if len(_bad_geom):
    print(
        f"\n!! {len(_bad_geom)} series have no usable PixelSpacing/Rows/Columns and will "
        "raise when loaded. Dropping them from the manifest -- an unscaled crop is worse "
        "than an absent slot."
    )
    MANIFEST = MANIFEST.drop(index=_bad_geom.index).reset_index(drop=True)

In [ ]:
from __future__ import annotations

# Size the chunk against the actual free space on /tmp, which Kaggle does not guarantee.
# This turns a mid-run ENOSPC into an up-front decision.
# 92 MB/study is the worst case: 64 slices x 512 x 512 uint8 x ~5.5 series.
import shutil as _sh

MB_PER_STUDY = 92
CACHE_DIR.mkdir(parents=True, exist_ok=True)
_du = _sh.disk_usage(CACHE_DIR)
_free_mb = _du.free / 1e6
# Use 60% of free space: the decode writes a temp file before os.replace, and something
# else on the box may want room too.
_fits = int(_free_mb * 0.60 / MB_PER_STUDY)

print(f"disk        : {CACHE_DIR} has {_free_mb / 1000:.1f} GB free of "
      f"{_du.total / 1e9:.1f} GB")
if _fits < 1:
    raise OSError(
        f"only {_free_mb / 1000:.1f} GB free under {CACHE_DIR}, which is not enough for "
        f"even one study at ~{MB_PER_STUDY} MB. submission.csv already holds the 0.5 "
        "fallback, so this run scores rather than errors -- but set CACHE_DIR to a "
        "roomier filesystem before trusting it."
    )
if _fits < CHUNK_STUDIES:
    print(f"  !! CHUNK_STUDIES reduced {CHUNK_STUDIES} -> {_fits} to fit free space")
    CHUNK_STUDIES = _fits
else:
    print(f"  CHUNK_STUDIES={CHUNK_STUDIES} fits (room for ~{_fits})")

CHUNKS = [TEST_UIDS[i : i + CHUNK_STUDIES] for i in range(0, N_TEST, CHUNK_STUDIES)]
print(f"\nchunks      : {len(CHUNKS)} x up to {CHUNK_STUDIES} studies")
print(f"peak cache  : ~{CHUNK_STUDIES * MB_PER_STUDY / 1000:.1f} GB under {CACHE_DIR} "
      "(built, scored and deleted one chunk at a time)")
ELAPSED = time.perf_counter() - T0
print(f"setup       : {ELAPSED:.0f}s of {NOTEBOOK_LIMIT_SECONDS}s, manifest {MANIFEST_SECONDS:.0f}s")


## 13. Score every checkpoint

Each checkpoint gets its **own** `StudyDataset` — the 3-slot cell and the 6-slot cell emit
different tensors — over the **same** shared slice cache. The slot list, `n_slices`, crop
and normalisation all come out of the checkpoint payload; nothing here is hardcoded.

Every dataset is built with `study_uids=TEST_UIDS`, so all members return the same studies
in the same order. `rank_blend` combines by position, so that is not cosmetic.

A per-checkpoint timing and a projection against the remaining budget is printed after each
member. With `BUDGET_GUARD` on, the loop stops early rather than being killed with nothing
written.

In [ ]:
from __future__ import annotations

def _find_checkpoint(rel: str) -> Path | None:
    """Resolve one CHECKPOINTS entry, whatever shape the upload ended up.

    Kaggle's CLI defaults to --dir-mode skip, so `kaggle datasets create -p data/models`
    uploads ONLY the two text files and silently drops every .pt. Other routes flatten
    the tree or add a wrapper directory. Filenames carry no meaning here -- slots, crop,
    normalisation and n_slices all come from inside the file -- so accepting any layout
    costs nothing and removes a whole class of silent mis-upload.
    """
    cell, _, base = rel.partition("/")
    exact = WEIGHTS_DIR / rel
    if exact.is_file():
        return exact
    flat = WEIGHTS_DIR / ("%s__%s" % (cell, base))      # cell name folded into filename
    if flat.is_file():
        return flat
    # Recursive: prefer a hit whose path still mentions the cell, so the two cells'
    # identically-named fold files can never be confused for one another.
    hits = sorted(WEIGHTS_DIR.rglob(base))
    tagged = [h for h in hits if cell in str(h)]
    if len(tagged) == 1:
        return tagged[0]
    if not tagged and len(hits) == 1 and len(set(c.split("/")[0] for c in CHECKPOINTS)) == 1:
        return hits[0]
    if len(tagged) > 1:
        raise FileNotFoundError(
            "%s is ambiguous under %s -- %d candidates:\n  %s"
            % (rel, WEIGHTS_DIR, len(tagged), "\n  ".join(str(h) for h in tagged))
        )
    return None


CHECKPOINT_PATHS = []
_absent = []
for _rel in CHECKPOINTS:
    _p = _find_checkpoint(_rel)
    if _p is not None:
        CHECKPOINT_PATHS.append(_p)
    else:
        _absent.append(_rel)

if _absent:
    _msg = (
        f"{len(_absent)} of {len(CHECKPOINTS)} checkpoints are not under {WEIGHTS_DIR}:\n  "
        + "\n  ".join(str(a) for a in _absent)
        + f"\nFound instead:\n  "
        + "\n  ".join(_dir_listing(str(WEIGHTS_DIR), 2)[:30])
    )
    if REQUIRE_ALL_CHECKPOINTS:
        raise FileNotFoundError(
            _msg + "\n\nSet REQUIRE_ALL_CHECKPOINTS=False to score a partial ensemble."
        )
    warnings.warn(_msg + "\n\nScoring a PARTIAL ensemble.", stacklevel=2)

if not CHECKPOINT_PATHS:
    raise FileNotFoundError("no checkpoints resolved; nothing to score")
print(f"{len(CHECKPOINT_PATHS)} checkpoint(s) resolved")
for _p in CHECKPOINT_PATHS:
    print(f"  {_p}  ({_p.stat().st_size / 1e6:.0f} MB)")
# A truncated upload is the failure that looks most like success. Every one of these is
# ~238 MB; anything materially smaller did not finish transferring.
_small = [p for p in CHECKPOINT_PATHS if p.stat().st_size < 200_000_000]
if _small:
    raise OSError(
        "these checkpoints are far smaller than the expected ~238 MB and are truncated "
        "uploads, not models:\n  "
        + "\n  ".join(f"{p} ({p.stat().st_size / 1e6:.0f} MB)" for p in _small)
    )

In [ ]:
from __future__ import annotations
import shutil

# One row per test study per member, in TEST_UIDS order. NaN means "never scored", which
# is distinguishable from a real 0.0 logit and is filled only at write time.
UID_POS = {uid: i for i, uid in enumerate(TEST_UIDS)}
MEMBER_LOGITS = {
    str(path): np.full((N_TEST, len(LABELS)), np.nan, dtype=np.float64)
    for path in CHECKPOINT_PATHS
}
SCORED_CHUNKS = 0
_chunk_seconds = []


def _blend_so_far() -> np.ndarray:
    """Per-column rank average over whatever has been scored, on the FULL test index.

    Ranking happens here, once, over every study -- never inside a chunk. Ranks computed
    within a chunk would not be comparable across chunks, and macro ROC-AUC scores exactly
    that global ordering.
    """
    usable = [a for a in MEMBER_LOGITS.values() if np.isfinite(a).any()]
    if not usable:
        return np.full((N_TEST, len(LABELS)), 0.5)
    # An unscored study gets the column's minimum so it ranks last rather than poisoning
    # the rank transform with a NaN.
    filled = []
    for a in usable:
        b = a.copy()
        for j in range(b.shape[1]):
            col = b[:, j]
            col[~np.isfinite(col)] = np.nanmin(col) if np.isfinite(col).any() else 0.0
        filled.append(b)
    return rank_blend(filled)


for _ci, _chunk in enumerate(CHUNKS):
    _remaining = NOTEBOOK_LIMIT_SECONDS - (time.perf_counter() - T0) - BUDGET_RESERVE_SECONDS
    if BUDGET_GUARD and _chunk_seconds and float(np.mean(_chunk_seconds)) > _remaining:
        print(
            f"\n!! BUDGET GUARD: next chunk projected at {np.mean(_chunk_seconds):.0f}s, "
            f"{_remaining:.0f}s remain. Stopping at {SCORED_CHUNKS}/{len(CHUNKS)} chunks "
            "and writing what is scored."
        )
        break

    _t0 = time.perf_counter()
    print(f"\n{'=' * 88}\nCHUNK {_ci + 1}/{len(CHUNKS)}  ({len(_chunk)} studies)\n{'=' * 88}")

    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    _rep = build_cache(
        SERIES_ROOT, CACHE_DIR, studies=_chunk, size=CACHE_SIZE, max_slices=MAX_SLICES,
        window=WINDOW_PERCENTILES, overwrite=CACHE_OVERWRITE,
        progress_every=CACHE_PROGRESS_EVERY,
    )
    print(f"cache      : {_rep.summary()}")
    if _rep.failures:
        print(f"             {len(_rep.failures)} series failed to decode -> absent slots")

    for _path in CHECKPOINT_PATHS:
        _ckpt = load_checkpoint(_path, device=DEVICE)
        with warnings.catch_warnings():
            warnings.simplefilter("once")
            _ds = _ckpt.dataset(MANIFEST, CACHE_DIR, study_uids=_chunk, verify_cache=True)
        _res = predict_studies(
            _ckpt.model, _ds, device=DEVICE, batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS, amp=AMP_RESOLVED, tta=TTA,
        )
        _target = MEMBER_LOGITS[str(_path)]
        for _uid, _row in zip(_res.study_uids, _res.scores):
            _target[UID_POS[_uid]] = _row
        print(f"  {_path.name:28s} {_res.seconds:6.1f}s  {len(_res.study_uids)} studies")
        del _ckpt, _ds
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    shutil.rmtree(CACHE_DIR, ignore_errors=True)   # evict before the next chunk
    SCORED_CHUNKS += 1
    _chunk_seconds.append(time.perf_counter() - _t0)

    if WRITE_PARTIAL_SUBMISSION:
        write_submission(OUT_PATH, TEST_UIDS, _blend_so_far())
        print(f"  partial submission refreshed ({SCORED_CHUNKS}/{len(CHUNKS)} chunks)")

    _elapsed = time.perf_counter() - T0
    _proj = _elapsed + float(np.mean(_chunk_seconds)) * (len(CHUNKS) - SCORED_CHUNKS)
    print(
        f"  budget     {_elapsed:.0f}s spent, projected total {_proj:.0f}s = "
        f"{100.0 * _proj / NOTEBOOK_LIMIT_SECONDS:.1f}% of {NOTEBOOK_LIMIT_SECONDS}s"
    )

_n_scored = int(np.isfinite(next(iter(MEMBER_LOGITS.values()))[:, 0]).sum())
print(f"\nscored {_n_scored} of {N_TEST} studies over {SCORED_CHUNKS}/{len(CHUNKS)} chunks "
      f"in {time.perf_counter() - T0:.0f}s")


## 14. Blend, write, validate

Per-column rank average across the members, then `write_submission`, then
`validate_submission`.

The validator re-reads the file off disk and runs the seven contract checks: header exact
(byte for byte), `Baker's` apostrophe intact, row count, UID order, 13 fields per row,
values finite, UIDs unique. It is the same set a constant-0.5 file passed at 470 bytes.

A check whose reference data was not supplied reports `----`, not `PASS` — a validator that
scores an unrun check as a pass is worse than no validator.

In [ ]:
from __future__ import annotations

_members = [a for a in MEMBER_LOGITS.values() if np.isfinite(a).any()]
if not _members:
    raise RuntimeError(
        "no member scored any study. submission.csv still holds the 0.5 fallback, which "
        "scores 0.5 rather than erroring -- but nothing here is a result."
    )

print(f"blending {len(_members)} of {len(MEMBER_LOGITS)} member(s) by PER-COLUMN RANK AVERAGE")
print("  (never a probability average: macro ROC-AUC scores only the within-column order)")

write_submission(OUT_PATH, TEST_UIDS, _blend_so_far())
print(f"\nwrote {OUT_PATH}")

CHECK = validate_submission(OUT_PATH, expected_uids=TEST_UIDS)
print(CHECK.summary())
if not CHECK.ok:
    # Do not raise: a contract-failing file still beats no file, and the checks above name
    # exactly what is wrong. Raising here would discard a scoreable submission.
    warnings.warn("submission failed a contract check -- see above", stacklevel=2)


In [ ]:
from __future__ import annotations

_sub = pd.read_csv(OUT_PATH)
print(f"shape  : {_sub.shape}   (expect ({N_TEST}, {len(SUBMISSION_COLUMNS)}))")
print(f"header : {list(_sub.columns)}")
assert list(_sub.columns) == list(SUBMISSION_COLUMNS), "pandas disagrees with the writer"
print()
print(_sub.head(10).to_string())
print()
print(_sub[list(LABELS)].describe().T[["min", "mean", "max"]].to_string())
print(f"\ntotal runtime: {time.perf_counter() - T0:.0f}s of {NOTEBOOK_LIMIT_SECONDS}s")
_sub.head()